# Mixture-of-Density: a CPU-only visual tutorial

This notebook visualizes the A4 / `marginal_cfm` algorithm implemented in Flow-Factory. It follows the pedagogical rhythm of [Rectified Flow](https://github.com/gnobitab/RectifiedFlow)—define a probability path, generate trajectory supervision, fit a velocity field, and animate generation—but studies a different outer-loop target:

\[
q_{k,t}(x)=(1-\alpha)\rho_{S_k,t}(x)+\alpha\rho_{T,t}(x).
\]

The notebook separates three objects throughout:

- **ideal target density** \(q_{k,t}\);
- **finite-capacity fitted velocity** \(\widehat v_{k+1}\);
- **actual generated density** \(\widehat\rho_{S_{k+1},t}\).

The fitted student is measured against the mixture; it is never assumed to equal it. Everything runs on CPU and all generated GIF/MP4/PNG/JSON artifacts go under `.scratch/mixture_of_density_demo/`.

## What does “a more complex teacher distribution” mean?

Complexity is a property of a distribution **relative to a representation and a student hypothesis class**; it is not the teacher's parameter count, and higher entropy alone is not necessarily harder. This demo uses several visible forms of complexity:

1. **Mode structure:** many persistent and rare high-density modes.
2. **High-density topology:** multiple separated superlevel components, rings, holes, and curved ridges. Gaussian mixtures still have full mathematical support; “disconnected” refers to high-density regions.
3. **Multiscale structure:** broad and narrow modes coexist.
4. **High-frequency detail:** sharp image edges and checkerboards that a smooth bottleneck student tends to lose.
5. **Conditional diversity:** several valid outputs for the same condition.

The 2D teacher is an analytic eight-mode Gaussian crown with alternating radii, widths, and weights. It has zero learned parameters but is deliberately difficult for the width-32 tanh student. The tiny-image section later translates these ideas into semantic templates and frequency content.

In [ ]:
from importlib.util import find_spec
import os
import time

NOTEBOOK_STARTED_AT = time.perf_counter()

_REQUIRED_MODULES = {
    "torch": "torch",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "PIL": "pillow",
    "imageio": "imageio[ffmpeg]",
    "imageio_ffmpeg": "imageio[ffmpeg]",
}
_missing = [package for module, package in _REQUIRED_MODULES.items() if find_spec(module) is None]
if _missing:
    raise RuntimeError(
        "Missing CPU demo dependencies: "
        f"{_missing!r}. Install with:\n"
        "  python -m pip install torch numpy matplotlib pillow 'imageio[ffmpeg]' jupyterlab ipykernel"
    )

from dataclasses import asdict, dataclass
from pathlib import Path
import copy
import json
import math
import random
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F

DEVICE = torch.device("cpu")
torch.set_default_dtype(torch.float32)


@dataclass(frozen=True)
class DemoConfig:
    fast_mode: bool = True
    alpha: float = 0.25
    seed: int = 20260801
    outer_iterations: int = 6
    inner_updates: int = 300
    cache_size: int = 4096
    batch_size: int = 256
    ode_steps: int = 24
    student_width: int = 32
    learning_rate: float = 2.0e-3
    visualization_samples: int = 2048
    metric_samples: int = 2048
    checkpoint_updates: Tuple[int, ...] = (0, 50, 150, 300)
    grid_limit: float = 3.8
    grid_bins: int = 96
    kde_bandwidth: float = 0.13
    sw_projections: int = 128
    export_fps: int = 6
    run_tiny_image: bool = False
    tiny_outer_iterations: int = 3
    tiny_inner_updates: int = 180
    tiny_cache_size: int = 2048
    tiny_batch_size: int = 256
    tiny_ode_steps: int = 16
    tiny_student_width: int = 32
    tiny_metric_samples_per_condition: int = 1536
    tiny_display_samples_per_condition: int = 16

    @classmethod
    def full(cls) -> "DemoConfig":
        return cls(
            fast_mode=False,
            outer_iterations=10,
            inner_updates=1200,
            cache_size=16384,
            batch_size=512,
            ode_steps=32,
            student_width=32,
            visualization_samples=10000,
            metric_samples=8192,
            checkpoint_updates=(0, 100, 400, 800, 1200),
            grid_bins=128,
            sw_projections=256,
            tiny_outer_iterations=4,
            tiny_inner_updates=500,
            tiny_cache_size=4096,
            tiny_metric_samples_per_condition=2048,
            tiny_display_samples_per_condition=16,
        )


MIXTURE_DENSITY_TEST_MODE = os.environ.get("MIXTURE_DENSITY_TEST_MODE", "0") == "1"
if MIXTURE_DENSITY_TEST_MODE:
    torch.set_num_threads(1)
FAST_MODE = True
CONFIG = DemoConfig() if FAST_MODE else DemoConfig.full()
OUTPUT_DIR = Path(".scratch/mixture_of_density_demo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(
    f"device={DEVICE}, fast_mode={CONFIG.fast_mode}, "
    f"test_mode={MIXTURE_DENSITY_TEST_MODE}, output_dir={OUTPUT_DIR}"
)

In [ ]:
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def cpu_generator(*keys: int) -> torch.Generator:
    """Create a deterministic CPU generator from integer namespace keys."""
    if not keys or any(isinstance(key, bool) or not isinstance(key, int) for key in keys):
        raise TypeError(f"expected one or more integer RNG keys, got {keys!r}")
    generator = torch.Generator(device="cpu")
    generator.manual_seed(hash(tuple(keys)) % (2**32))
    return generator


def figure_to_rgb(fig: plt.Figure) -> np.ndarray:
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba())
    return np.ascontiguousarray(rgba[..., :3])


def save_frames(frames: Sequence[np.ndarray], stem: str, fps: int) -> Dict[str, Path]:
    if not frames:
        raise ValueError(f"expected at least one frame for {stem!r}")
    paths = {
        "gif": OUTPUT_DIR / f"{stem}.gif",
        "mp4": OUTPUT_DIR / f"{stem}.mp4",
    }
    gif_duration_seconds = 1.0 / fps
    imageio.mimsave(
        paths["gif"], frames, format="GIF", duration=gif_duration_seconds, loop=0
    )
    initial_gif_timing = inspect_gif_timing(paths["gif"])
    if len(frames) > 1 and (
        not initial_gif_timing["timing_available"]
        or abs(
            initial_gif_timing["mean_frame_duration_seconds"] - gif_duration_seconds
        )
        > 0.02
    ):
        # Some ImageIO/Pillow combinations interpret the ImageIO seconds value as
        # Pillow milliseconds. Rewrite only the container timing with Pillow's
        # documented millisecond API while preserving the requested seconds value.
        pil_frames = [Image.fromarray(frame) for frame in frames]
        pil_frames[0].save(
            paths["gif"],
            save_all=True,
            append_images=pil_frames[1:],
            duration=max(10, int(round(1000.0 * gif_duration_seconds))),
            loop=0,
        )
    try:
        imageio.mimsave(paths["mp4"], frames, fps=fps, codec="libx264", quality=8)
    except (ImportError, ModuleNotFoundError, ValueError, RuntimeError, OSError) as exc:
        print(
            f"MP4 export skipped for {stem!r}: ffmpeg backend unavailable or export failed: "
            f"{type(exc).__name__}: {exc}"
        )
        paths.pop("mp4")
    return paths


set_all_seeds(CONFIG.seed)
with (OUTPUT_DIR / "config.json").open("w", encoding="utf-8") as handle:
    json.dump(asdict(CONFIG), handle, indent=2)

print("reproducibility helpers ready")

## Mixture-of-Density training rule

At outer iteration \(k\), freeze the current student and define

\[
q_{k,t}=(1-\alpha)\rho_{S_k,t}+\alpha\rho_{T,t},
\qquad
q_{k,t}u_{k,t}=(1-\alpha)\rho_{S_k,t}v_k+\alpha\rho_{T,t}v_T.
\]

The conceptual density responsibility is

\[
\gamma_{k,t}(x)=\frac{\alpha\rho_{T,t}(x)}{q_{k,t}(x)},
\qquad
u_{k,t}(x)=(1-\gamma_{k,t}(x))v_k(t,x)+\gamma_{k,t}(x)v_T(t,x).
\]

Training does **not** evaluate this ratio. Instead, draw one branch for the complete trajectory:

\[
B\sim\operatorname{Bernoulli}(\alpha),\quad
(X_t,V_t)=
\begin{cases}
(\Phi_t^{v_k}(Z),v_k(t,X_t)),&B=0,\\
(\Phi_t^{v_T}(Z),v_T(t,X_t)),&B=1,
\end{cases}
\]

and fit the detached source label:

\[
\mathcal L_k(\theta)=\mathbb E\|v_\theta(t,X_t)-V_t\|_2^2.
\]

The population loss equals Flow Matching to \(u_{k,t}\) plus an irreducible branch-label variance. Therefore the raw CFM loss need not approach zero. With an exact fit, the endpoint recursion would be

\[
\rho^{\mathrm{ideal}}_{S_k,1}=(1-\alpha)^k\rho_{S_0,1}+[1-(1-\alpha)^k]\rho_{T,1}.
\]

The notebook plots this ideal recursion only as a reference; the trained finite MLP is evaluated rather than assumed exact.

## Analytic eight-component Gaussian-crown teacher

Let the base be \(Z\sim\mathcal N(0,I_2)\). Draw a component \(C=j\) with probability \(w_j\), then draw the endpoint independently as

\[
Y\mid C=j\sim\mathcal N(\mu_j,\sigma_j^2 I_2),
\qquad
X_t=(1-t)Z+tY,\quad 0\le t\le1.
\]

The eight means lie at equally spaced angles \(\theta_j=2\pi j/8\), but alternate radii \(r_j\in\{1.7,2.5\}\):
\(\mu_j=r_j(\cos\theta_j,\sin\theta_j)\). Their unequal weights are
\((0.18,0.07,0.15,0.08,0.17,0.06,0.17,0.12)\), and their widths alternate between \(0.13\) and \(0.22\). This creates common and rare modes at two spatial and density scales.

Conditioned on component \(j\), the path marginal remains Gaussian:

\[
X_t\mid C=j\sim\mathcal N(m_j(t),s_j^2(t)I_2),\qquad
m_j(t)=t\mu_j,\qquad
s_j^2(t)=(1-t)^2+t^2\sigma_j^2.
\]

Writing \(\varphi_2(x;m,s^2I)\) for the normalized two-dimensional Gaussian density, the exact component density and responsibility are

\[
p_{j,t}(x)=\varphi_2(x;t\mu_j,s_j^2(t)I_2),\qquad
r_{j,t}(x)=\frac{w_jp_{j,t}(x)}{\sum_{\ell=1}^{8}w_\ell p_{\ell,t}(x)}.
\]

For the linear path, the conditional velocity is \(Y-Z\). Gaussian conditioning gives the component marginal velocity

\[
v_j(t,x)=\mathbb E[Y-Z\mid X_t=x,C=j]
=\mu_j+\frac{t\sigma_j^2-(1-t)}{s_j^2(t)}(x-t\mu_j).
\]

Therefore the analytic teacher velocity is the responsibility-weighted field

\[
v_T(t,x)=\sum_{j=1}^{8}r_{j,t}(x)v_j(t,x).
\]

Every Gaussian component has positive density everywhere, so the mixture has full support on \(\mathbb R^2\). The visually separated “crown modes” refer to disconnected or nearly disconnected **high-density** superlevel regions, not disconnected mathematical support.

### Discrete-time supervision in this demo

The source cache stores the fixed RK4 grid, including both endpoints, and the inner fit samples uniformly from those stored times. Its objective is therefore a discrete quadrature approximation with explicit endpoint mass, unlike standard continuous-time Flow Matching that samples \(t\) from a continuous distribution. Increasing `ode_steps` refines this approximation.

Inner checkpoints are cloned model `state_dict` snapshots for later visualization only. They intentionally do **not** contain optimizer state or RNG state and are not resumable training checkpoints.

In [ ]:
class TeacherGaussianCrown:
    """Represent the analytic two-dimensional Gaussian-crown probability path."""

    def __init__(
        self,
        weights: Sequence[float] = (0.18, 0.07, 0.15, 0.08, 0.17, 0.06, 0.17, 0.12),
        radii: Tuple[float, float] = (1.7, 2.5),
        widths: Tuple[float, float] = (0.13, 0.22),
    ) -> None:
        weights_tensor = torch.as_tensor(weights, dtype=torch.float32, device=DEVICE)
        if weights_tensor.ndim != 1 or weights_tensor.numel() < 2:
            raise ValueError(
                f"expected one-dimensional weights with at least two modes, got shape "
                f"{tuple(weights_tensor.shape)}"
            )
        if not torch.isfinite(weights_tensor).all() or (weights_tensor <= 0).any():
            raise ValueError(f"expected finite positive weights, got {weights_tensor.tolist()!r}")
        if len(radii) != 2 or not all(math.isfinite(value) and value > 0 for value in radii):
            raise ValueError(f"expected two finite positive alternating radii, got {radii!r}")
        if len(widths) != 2 or not all(math.isfinite(value) and value > 0 for value in widths):
            raise ValueError(f"expected two finite positive alternating widths, got {widths!r}")

        self.weights = weights_tensor / weights_tensor.sum()
        mode_indices = torch.arange(weights_tensor.numel(), dtype=torch.float32, device=DEVICE)
        angles = 2.0 * math.pi * mode_indices / weights_tensor.numel()
        alternating_radii = torch.tensor(radii, dtype=torch.float32)[
            torch.arange(weights_tensor.numel()) % 2
        ]
        self.means = alternating_radii[:, None] * torch.stack(
            (torch.cos(angles), torch.sin(angles)), dim=-1
        )
        self.widths = torch.tensor(widths, dtype=torch.float32)[
            torch.arange(weights_tensor.numel()) % 2
        ]
        self.state_dim = 2
        self.num_components = weights_tensor.numel()

    @staticmethod
    def _validate_sample_count(sample_count: int) -> None:
        if isinstance(sample_count, bool) or not isinstance(sample_count, int) or sample_count <= 0:
            raise ValueError(f"expected sample_count to be a positive integer, got {sample_count!r}")

    def _prepare_time(self, time: object, batch_size: int) -> torch.Tensor:
        time_tensor = torch.as_tensor(time, dtype=torch.float32, device=DEVICE)
        if time_tensor.ndim == 0:
            time_tensor = time_tensor.expand(batch_size).reshape(batch_size, 1)
        elif time_tensor.ndim == 1 and time_tensor.shape[0] == batch_size:
            time_tensor = time_tensor[:, None]
        elif time_tensor.ndim == 2 and time_tensor.shape == (batch_size, 1):
            pass
        else:
            raise ValueError(
                f"expected scalar time or shape ({batch_size},) / ({batch_size}, 1), "
                f"got {tuple(time_tensor.shape)}"
            )
        if not torch.isfinite(time_tensor).all() or (time_tensor < 0).any() or (time_tensor > 1).any():
            raise ValueError(f"expected finite times in [0, 1], got {time_tensor.flatten().tolist()!r}")
        return time_tensor

    def _component_terms(
        self, time: object, state: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        if not isinstance(state, torch.Tensor):
            raise TypeError(f"expected state to be torch.Tensor, got {type(state).__name__}")
        if state.device.type != "cpu" or state.ndim != 2 or state.shape[1] != self.state_dim:
            raise ValueError(
                f"expected CPU state shape (batch, {self.state_dim}), got device={state.device}, "
                f"shape={tuple(state.shape)}"
            )
        state = state.to(dtype=torch.float32)
        if not torch.isfinite(state).all():
            raise ValueError("expected finite state values for Gaussian-crown evaluation")

        time_tensor = self._prepare_time(time, state.shape[0])
        component_variances = (1.0 - time_tensor).square() + time_tensor.square() * self.widths.square()
        component_means = time_tensor[:, None, :] * self.means[None, :, :]
        residuals = state[:, None, :] - component_means
        log_component_densities = -math.log(2.0 * math.pi) - torch.log(component_variances)
        log_component_densities = log_component_densities - (
            residuals.square().sum(dim=-1) / (2.0 * component_variances)
        )
        return time_tensor, residuals, log_component_densities

    def sample_base(
        self, sample_count: int, generator: Optional[torch.Generator] = None
    ) -> torch.Tensor:
        """Draw base samples from the standard two-dimensional Gaussian."""
        self._validate_sample_count(sample_count)
        return torch.randn(sample_count, self.state_dim, generator=generator, device=DEVICE)

    def sample_endpoint(
        self, sample_count: int, generator: Optional[torch.Generator] = None
    ) -> torch.Tensor:
        """Draw endpoint samples directly from the crown mixture."""
        self._validate_sample_count(sample_count)
        components = torch.multinomial(
            self.weights, sample_count, replacement=True, generator=generator
        )
        noise = torch.randn(sample_count, self.state_dim, generator=generator, device=DEVICE)
        return self.means[components] + self.widths[components, None] * noise

    def density(self, time: object, state: torch.Tensor) -> torch.Tensor:
        """Evaluate the normalized path-marginal mixture density."""
        _, _, log_component_densities = self._component_terms(time, state)
        log_weighted = torch.log(self.weights)[None, :] + log_component_densities
        return torch.exp(torch.logsumexp(log_weighted, dim=-1))

    def responsibilities(self, time: object, state: torch.Tensor) -> torch.Tensor:
        """Evaluate exact posterior component responsibilities."""
        _, _, log_component_densities = self._component_terms(time, state)
        return torch.softmax(torch.log(self.weights)[None, :] + log_component_densities, dim=-1)

    def velocity(self, time: object, state: torch.Tensor) -> torch.Tensor:
        """Evaluate the exact marginal velocity of the Gaussian-crown path."""
        time_tensor, residuals, log_component_densities = self._component_terms(time, state)
        component_variances = (1.0 - time_tensor).square() + time_tensor.square() * self.widths.square()
        coefficients = (
            time_tensor * self.widths.square() - (1.0 - time_tensor)
        ) / component_variances
        component_velocities = self.means[None, :, :] + coefficients[:, :, None] * residuals
        responsibilities = torch.softmax(
            torch.log(self.weights)[None, :] + log_component_densities, dim=-1
        )
        velocity = (responsibilities[:, :, None] * component_velocities).sum(dim=1)
        if not torch.isfinite(velocity).all():
            raise ValueError("analytic Gaussian-crown velocity produced non-finite values")
        return velocity


TEACHER_2D = TeacherGaussianCrown()

In [ ]:
class VelocityMLP(nn.Module):
    """Predict a velocity from state, time features, and optional conditions."""

    def __init__(self, state_dim: int, condition_dim: int = 0, width: int = 32) -> None:
        super().__init__()
        for name, value in {
            "state_dim": state_dim,
            "condition_dim": condition_dim,
            "width": width,
        }.items():
            if isinstance(value, bool) or not isinstance(value, int):
                raise TypeError(f"expected integer {name}, got {type(value).__name__}: {value!r}")
        if state_dim <= 0 or condition_dim < 0 or width <= 0:
            raise ValueError(
                f"expected state_dim>0, condition_dim>=0, width>0; got "
                f"state_dim={state_dim}, condition_dim={condition_dim}, width={width}"
            )
        self.state_dim = state_dim
        self.condition_dim = condition_dim
        self.width = width
        self.network = nn.Sequential(
            nn.Linear(state_dim + condition_dim + 3, width),
            nn.Tanh(),
            nn.Linear(width, width),
            nn.Tanh(),
            nn.Linear(width, state_dim),
        )
        nn.init.zeros_(self.network[-1].weight)
        nn.init.zeros_(self.network[-1].bias)

    @staticmethod
    def _time_features(time: object, batch_size: int) -> torch.Tensor:
        time_tensor = torch.as_tensor(time, dtype=torch.float32, device=DEVICE)
        if time_tensor.ndim == 0:
            time_tensor = time_tensor.expand(batch_size).reshape(batch_size, 1)
        elif time_tensor.ndim == 1 and time_tensor.shape[0] == batch_size:
            time_tensor = time_tensor[:, None]
        elif time_tensor.ndim == 2 and time_tensor.shape == (batch_size, 1):
            pass
        else:
            raise ValueError(
                f"expected scalar time or shape ({batch_size},) / ({batch_size}, 1), "
                f"got {tuple(time_tensor.shape)}"
            )
        if not torch.isfinite(time_tensor).all() or (time_tensor < 0).any() or (time_tensor > 1).any():
            raise ValueError("expected finite model times in [0, 1]")
        return torch.cat(
            (time_tensor, torch.sin(math.pi * time_tensor), torch.cos(math.pi * time_tensor)),
            dim=-1,
        )

    def forward(
        self,
        time: object,
        state: torch.Tensor,
        condition: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Predict batched velocities in FP32 on CPU."""
        if not isinstance(state, torch.Tensor):
            raise TypeError(f"expected state to be torch.Tensor, got {type(state).__name__}")
        if state.device.type != "cpu" or state.ndim != 2 or state.shape[1] != self.state_dim:
            raise ValueError(
                f"expected CPU state shape (batch, {self.state_dim}), got device={state.device}, "
                f"shape={tuple(state.shape)}"
            )
        state = state.to(dtype=torch.float32)
        if not torch.isfinite(state).all():
            raise ValueError("expected finite state values for velocity prediction")
        time_features = self._time_features(time, state.shape[0])

        if self.condition_dim == 0:
            if condition is not None:
                raise ValueError("expected condition=None because condition_dim=0")
            features = torch.cat((state, time_features), dim=-1)
        else:
            if not isinstance(condition, torch.Tensor):
                raise TypeError(
                    f"expected condition torch.Tensor for condition_dim={self.condition_dim}, "
                    f"got {type(condition).__name__}"
                )
            if (
                condition.device.type != "cpu"
                or condition.ndim != 2
                or condition.shape != (state.shape[0], self.condition_dim)
            ):
                raise ValueError(
                    f"expected CPU condition shape ({state.shape[0]}, {self.condition_dim}), "
                    f"got device={condition.device}, shape={tuple(condition.shape)}"
                )
            condition = condition.to(dtype=torch.float32)
            if not torch.isfinite(condition).all():
                raise ValueError("expected finite condition values for velocity prediction")
            features = torch.cat((state, time_features, condition), dim=-1)
        velocity = self.network(features)
        if not torch.isfinite(velocity).all():
            raise ValueError("VelocityMLP produced non-finite values")
        return velocity


def clone_frozen_model(model: nn.Module) -> nn.Module:
    """Clone a model onto CPU and freeze all of its parameters."""
    if not isinstance(model, nn.Module):
        raise TypeError(f"expected nn.Module to clone, got {type(model).__name__}")
    frozen_model = copy.deepcopy(model).to(device=DEVICE, dtype=torch.float32)
    frozen_model.eval()
    frozen_model.requires_grad_(False)
    return frozen_model


def _evaluate_velocity(
    velocity_model: object,
    time: torch.Tensor,
    state: torch.Tensor,
    condition: Optional[torch.Tensor],
) -> torch.Tensor:
    velocity = (
        velocity_model(time, state)
        if condition is None
        else velocity_model(time, state, condition)
    )
    if not isinstance(velocity, torch.Tensor) or velocity.shape != state.shape:
        received = type(velocity).__name__ if not isinstance(velocity, torch.Tensor) else tuple(velocity.shape)
        raise ValueError(
            f"expected velocity tensor shape {tuple(state.shape)}, got {received!r}"
        )
    velocity = velocity.to(dtype=torch.float32)
    if not torch.isfinite(velocity).all():
        raise ValueError(f"velocity evaluation at time={float(time):.6f} produced non-finite values")
    return velocity


def integrate_fixed_rk4(
    velocity_model: object,
    initial_state: torch.Tensor,
    steps: int,
    condition: Optional[torch.Tensor] = None,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Integrate a velocity field with fixed-step RK4 and retain every stored time."""
    if isinstance(steps, bool) or not isinstance(steps, int) or steps <= 0:
        raise ValueError(f"expected steps to be a positive integer, got {steps!r}")
    if not callable(velocity_model):
        raise TypeError(f"expected callable velocity_model, got {type(velocity_model).__name__}")
    if not isinstance(initial_state, torch.Tensor):
        raise TypeError(
            f"expected initial_state to be torch.Tensor, got {type(initial_state).__name__}"
        )
    if initial_state.device.type != "cpu" or initial_state.ndim != 2:
        raise ValueError(
            f"expected CPU initial_state shape (batch, state_dim), got "
            f"device={initial_state.device}, shape={tuple(initial_state.shape)}"
        )
    if initial_state.shape[0] == 0 or initial_state.shape[1] == 0:
        raise ValueError(f"expected non-empty initial_state, got shape {tuple(initial_state.shape)}")
    state = initial_state.to(dtype=torch.float32)
    if not torch.isfinite(state).all():
        raise ValueError("expected finite initial_state values")
    if condition is not None:
        if not isinstance(condition, torch.Tensor) or condition.device.type != "cpu":
            raise TypeError("expected condition to be a CPU torch.Tensor when provided")
        if condition.ndim != 2 or condition.shape[0] != state.shape[0]:
            raise ValueError(
                f"expected condition shape ({state.shape[0]}, condition_dim), got "
                f"{tuple(condition.shape)}"
            )
        condition = condition.to(dtype=torch.float32)
        if not torch.isfinite(condition).all():
            raise ValueError("expected finite condition values")

    times = torch.linspace(0.0, 1.0, steps + 1, device=DEVICE)
    step_size = 1.0 / steps
    stored_states = [state]
    for step_index in range(steps):
        time = times[step_index]
        midpoint = time + 0.5 * step_size
        k1 = _evaluate_velocity(velocity_model, time, state, condition)
        k2 = _evaluate_velocity(
            velocity_model, midpoint, state + 0.5 * step_size * k1, condition
        )
        k3 = _evaluate_velocity(
            velocity_model, midpoint, state + 0.5 * step_size * k2, condition
        )
        k4 = _evaluate_velocity(velocity_model, time + step_size, state + step_size * k3, condition)
        state = state + (step_size / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
        if not torch.isfinite(state).all():
            raise ValueError(f"RK4 produced non-finite state after step {step_index + 1}/{steps}")
        stored_states.append(state)

    trajectories = torch.stack(stored_states, dim=1)
    stored_velocities = torch.stack(
        [
            _evaluate_velocity(velocity_model, times[index], trajectories[:, index], condition)
            for index in range(steps + 1)
        ],
        dim=1,
    )
    return times, trajectories, stored_velocities

In [ ]:
@dataclass(frozen=True)
class SourceTrajectoryCache:
    """Store complete source trajectories and their detached supervision labels."""

    times: torch.Tensor
    states: torch.Tensor
    velocities: torch.Tensor
    teacher_branch: torch.Tensor

    def __post_init__(self) -> None:
        if self.times.ndim != 1:
            raise ValueError(f"expected times shape (stored_times,), got {tuple(self.times.shape)}")
        if self.states.ndim != 3 or self.velocities.shape != self.states.shape:
            raise ValueError(
                f"expected matching states/velocities shape (trajectories, times, state_dim), "
                f"got states={tuple(self.states.shape)}, velocities={tuple(self.velocities.shape)}"
            )
        if self.states.shape[1] != self.times.numel():
            raise ValueError(
                f"expected {self.times.numel()} stored state times, got {self.states.shape[1]}"
            )
        if self.teacher_branch.dtype != torch.bool or self.teacher_branch.shape != (
            self.states.shape[0],
        ):
            raise ValueError(
                f"expected bool teacher_branch shape ({self.states.shape[0]},), got "
                f"dtype={self.teacher_branch.dtype}, shape={tuple(self.teacher_branch.shape)}"
            )
        if not (
            torch.isfinite(self.times).all()
            and torch.isfinite(self.states).all()
            and torch.isfinite(self.velocities).all()
        ):
            raise ValueError("source cache contains non-finite times, states, or velocities")


def generate_source_cache(
    old_student: nn.Module,
    teacher: TeacherGaussianCrown,
    cache_size: int,
    ode_steps: int,
    alpha: float,
    seed: int,
) -> SourceTrajectoryCache:
    """Generate one fixed old-student/teacher branch trajectory per base sample."""
    if isinstance(cache_size, bool) or not isinstance(cache_size, int) or cache_size <= 1:
        raise ValueError(f"expected cache_size integer greater than one, got {cache_size!r}")
    if isinstance(ode_steps, bool) or not isinstance(ode_steps, int) or ode_steps <= 0:
        raise ValueError(f"expected ode_steps to be a positive integer, got {ode_steps!r}")
    if isinstance(seed, bool) or not isinstance(seed, int):
        raise TypeError(f"expected integer cache seed, got {type(seed).__name__}: {seed!r}")
    if isinstance(alpha, bool) or not isinstance(alpha, (int, float)):
        raise TypeError(f"expected numeric alpha, got {type(alpha).__name__}: {alpha!r}")
    if not math.isfinite(float(alpha)) or not 0.0 <= float(alpha) <= 1.0:
        raise ValueError(f"expected finite alpha in [0, 1], got {alpha!r}")
    if not isinstance(teacher, TeacherGaussianCrown):
        raise TypeError(f"expected TeacherGaussianCrown, got {type(teacher).__name__}")

    frozen_old_student = clone_frozen_model(old_student)
    base_samples = teacher.sample_base(cache_size, cpu_generator(seed, 101))
    teacher_branch = torch.rand(cache_size, generator=cpu_generator(seed, 102)) < float(alpha)
    stored_times = torch.linspace(0.0, 1.0, ode_steps + 1, device=DEVICE)
    states = torch.empty(cache_size, ode_steps + 1, teacher.state_dim, device=DEVICE)
    velocities = torch.empty_like(states)

    with torch.no_grad():
        old_mask = ~teacher_branch
        if old_mask.any():
            old_times, old_states, old_velocities = integrate_fixed_rk4(
                frozen_old_student, base_samples[old_mask], ode_steps
            )
            states[old_mask] = old_states
            velocities[old_mask] = old_velocities
            stored_times = old_times
        if teacher_branch.any():
            teacher_times, teacher_states, teacher_velocities = integrate_fixed_rk4(
                teacher.velocity, base_samples[teacher_branch], ode_steps
            )
            states[teacher_branch] = teacher_states
            velocities[teacher_branch] = teacher_velocities
            stored_times = teacher_times

    return SourceTrajectoryCache(
        times=stored_times.detach(),
        states=states.detach(),
        velocities=velocities.detach(),
        teacher_branch=teacher_branch.detach(),
    )

In [ ]:
@dataclass
class StudentFitResult:
    """Collect one fit; checkpoint state_dicts are visualization-only snapshots."""

    model: nn.Module
    history: List[Dict[str, float]]
    checkpoints: Dict[int, Dict[str, torch.Tensor]]
    best_update: int
    stopped_early: bool


def _cloned_cpu_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def _event_mean_mse(prediction: torch.Tensor, label: torch.Tensor) -> torch.Tensor:
    if prediction.shape != label.shape or prediction.ndim != 2:
        raise ValueError(
            f"expected matching event tensors shape (batch, state_dim), got "
            f"prediction={tuple(prediction.shape)}, label={tuple(label.shape)}"
        )
    return (prediction.float() - label.detach().float()).square().mean(dim=-1).mean()


def fit_student_from_cache(
    old_student: nn.Module,
    cache: SourceTrajectoryCache,
    updates: int,
    batch_size: int,
    learning_rate: float,
    seed: int,
    checkpoint_updates: Sequence[int] = (),
    validation_fraction: float = 0.15,
    early_stopping_patience: int = 50,
    early_stopping_min_delta: float = 1.0e-6,
    max_updates: Optional[int] = None,
) -> StudentFitResult:
    """Warm-start a student with detached cached labels and validation early stopping."""
    for name, value in {"updates": updates, "batch_size": batch_size}.items():
        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
            raise ValueError(f"expected positive integer {name}, got {value!r}")
    if max_updates is None:
        hard_cap = updates
    elif isinstance(max_updates, bool) or not isinstance(max_updates, int) or max_updates <= 0:
        raise ValueError(f"expected max_updates=None or a positive integer, got {max_updates!r}")
    else:
        hard_cap = min(updates, max_updates)
    if isinstance(seed, bool) or not isinstance(seed, int):
        raise TypeError(f"expected integer training seed, got {type(seed).__name__}: {seed!r}")
    if not math.isfinite(learning_rate) or learning_rate <= 0:
        raise ValueError(f"expected finite positive learning_rate, got {learning_rate!r}")
    if not math.isfinite(validation_fraction) or not 0.0 < validation_fraction < 0.5:
        raise ValueError(
            f"expected validation_fraction in (0, 0.5), got {validation_fraction!r}"
        )
    if (
        isinstance(early_stopping_patience, bool)
        or not isinstance(early_stopping_patience, int)
        or early_stopping_patience <= 0
    ):
        raise ValueError(
            f"expected positive integer early_stopping_patience, got {early_stopping_patience!r}"
        )
    if not math.isfinite(early_stopping_min_delta) or early_stopping_min_delta < 0:
        raise ValueError(
            f"expected finite nonnegative early_stopping_min_delta, got "
            f"{early_stopping_min_delta!r}"
        )

    trajectory_count, stored_time_count, _ = cache.states.shape
    validation_count = max(1, int(round(trajectory_count * validation_fraction)))
    if validation_count >= trajectory_count:
        raise ValueError(
            f"validation split leaves no training trajectories: cache_size={trajectory_count}, "
            f"validation_count={validation_count}"
        )
    split_generator = cpu_generator(seed, 201)
    permutation = torch.randperm(trajectory_count, generator=split_generator)
    validation_trajectories = permutation[:validation_count]
    training_trajectories = permutation[validation_count:]
    validation_times = torch.randint(
        stored_time_count,
        (validation_count,),
        generator=cpu_generator(seed, 202),
    )

    student = copy.deepcopy(old_student).to(device=DEVICE, dtype=torch.float32)
    student.train()
    student.requires_grad_(True)
    optimizer = torch.optim.Adam(student.parameters(), lr=float(learning_rate))
    requested_checkpoints = {int(update) for update in checkpoint_updates if 0 <= int(update) <= hard_cap}
    requested_checkpoints.update((0, hard_cap))
    checkpoints = {0: _cloned_cpu_state_dict(student)}
    validation_states = cache.states[validation_trajectories, validation_times]
    validation_labels = cache.velocities[validation_trajectories, validation_times].detach()
    student.eval()
    with torch.no_grad():
        initial_validation_prediction = student(
            cache.times[validation_times], validation_states
        )
        initial_validation_loss = _event_mean_mse(
            initial_validation_prediction, validation_labels
        )
    if not torch.isfinite(initial_validation_loss):
        raise ValueError(
            f"non-finite validation loss at warm-start update 0: "
            f"{initial_validation_loss.item()!r}"
        )

    history: List[Dict[str, float]] = [
        {
            "update": 0.0,
            "validation_loss": float(initial_validation_loss),
        }
    ]
    best_validation_loss = float(initial_validation_loss)
    best_update = 0
    best_state = _cloned_cpu_state_dict(student)
    stale_updates = 0
    stopped_early = False
    training_generator = cpu_generator(seed, 203)
    student.train()

    for update in range(1, hard_cap + 1):
        sampled_training_positions = torch.randint(
            training_trajectories.numel(),
            (batch_size,),
            generator=training_generator,
        )
        sampled_trajectories = training_trajectories[sampled_training_positions]
        sampled_times = torch.randint(
            stored_time_count,
            (batch_size,),
            generator=training_generator,
        )
        training_states = cache.states[sampled_trajectories, sampled_times]
        training_labels = cache.velocities[sampled_trajectories, sampled_times].detach()
        training_time_values = cache.times[sampled_times]

        optimizer.zero_grad(set_to_none=True)
        training_prediction = student(training_time_values, training_states)
        training_loss = _event_mean_mse(training_prediction, training_labels)
        if not torch.isfinite(training_loss):
            raise ValueError(f"non-finite training loss at inner update {update}: {training_loss.item()!r}")
        training_loss.backward()
        optimizer.step()

        student.eval()
        with torch.no_grad():
            validation_states = cache.states[validation_trajectories, validation_times]
            validation_labels = cache.velocities[
                validation_trajectories, validation_times
            ].detach()
            validation_prediction = student(
                cache.times[validation_times], validation_states
            )
            validation_loss = _event_mean_mse(validation_prediction, validation_labels)
        if not torch.isfinite(validation_loss):
            raise ValueError(
                f"non-finite validation loss at inner update {update}: "
                f"{validation_loss.item()!r}"
            )
        history.append(
            {
                "update": float(update),
                "train_loss": float(training_loss.detach()),
                "validation_loss": float(validation_loss),
            }
        )

        if update in requested_checkpoints:
            checkpoints[update] = _cloned_cpu_state_dict(student)
        if float(validation_loss) < best_validation_loss - early_stopping_min_delta:
            best_validation_loss = float(validation_loss)
            best_update = update
            best_state = _cloned_cpu_state_dict(student)
            stale_updates = 0
        else:
            stale_updates += 1
            if stale_updates >= early_stopping_patience:
                stopped_early = True
                if update not in checkpoints:
                    checkpoints[update] = _cloned_cpu_state_dict(student)
                break
        student.train()

    student.load_state_dict(best_state)
    student.eval()
    checkpoints[best_update] = _cloned_cpu_state_dict(student)
    return StudentFitResult(
        model=student,
        history=history,
        checkpoints=checkpoints,
        best_update=best_update,
        stopped_early=stopped_early,
    )

In [ ]:
@dataclass
class OuterIterationRecord:
    """Store one outer iteration without plotting or distributional metrics."""

    outer_iteration: int
    ideal_accumulated_teacher_weight: float
    seeds: Dict[str, int]
    old_trajectory: Tuple[torch.Tensor, torch.Tensor, torch.Tensor]
    teacher_trajectory: Tuple[torch.Tensor, torch.Tensor, torch.Tensor]
    actual_trajectory: Tuple[torch.Tensor, torch.Tensor, torch.Tensor]
    branch_cache_summary: Dict[str, float]
    train_history: List[Dict[str, float]]
    checkpoints: Dict[int, Dict[str, torch.Tensor]]
    best_update: int
    stopped_early: bool


@dataclass
class OuterLoopResult:
    """Collect the final frozen student and all reproducible outer-loop records."""

    final_student: nn.Module
    iterations: List[OuterIterationRecord]


def _detached_trajectory(
    trajectory: Tuple[torch.Tensor, torch.Tensor, torch.Tensor]
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    return tuple(value.detach().cpu().clone() for value in trajectory)


def run_mixture_of_density_outer_loop(
    teacher: TeacherGaussianCrown,
    alpha: float,
    outer_iterations: int,
    cache_size: int,
    inner_updates: int,
    batch_size: int,
    ode_steps: int,
    student_width: int,
    learning_rate: float,
    seed: int,
    visualization_samples: int = 256,
    checkpoint_updates: Sequence[int] = (),
    initial_student: Optional[VelocityMLP] = None,
    early_stopping_patience: int = 50,
) -> OuterLoopResult:
    """Fit and freeze successive students while retaining reproducible numeric records."""
    for name, value in {
        "outer_iterations": outer_iterations,
        "cache_size": cache_size,
        "inner_updates": inner_updates,
        "batch_size": batch_size,
        "ode_steps": ode_steps,
        "student_width": student_width,
        "visualization_samples": visualization_samples,
    }.items():
        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
            raise ValueError(f"expected positive integer {name}, got {value!r}")
    if isinstance(seed, bool) or not isinstance(seed, int):
        raise TypeError(f"expected integer outer-loop seed, got {type(seed).__name__}: {seed!r}")
    if not math.isfinite(float(alpha)) or not 0.0 <= float(alpha) <= 1.0:
        raise ValueError(f"expected finite alpha in [0, 1], got {alpha!r}")

    if initial_student is None:
        initialization_seed = cpu_generator(seed, 301).initial_seed()
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(initialization_seed)
            current_student: nn.Module = VelocityMLP(
                state_dim=teacher.state_dim,
                condition_dim=0,
                width=student_width,
            )
    else:
        if initial_student.state_dim != teacher.state_dim or initial_student.condition_dim != 0:
            raise ValueError(
                f"expected unconditional initial student with state_dim={teacher.state_dim}, got "
                f"state_dim={initial_student.state_dim}, condition_dim={initial_student.condition_dim}"
            )
        current_student = copy.deepcopy(initial_student)
    current_student = clone_frozen_model(current_student)
    records: List[OuterIterationRecord] = []

    for outer_index in range(outer_iterations):
        old_student = clone_frozen_model(current_student)
        iteration_seeds = {
            "cache": seed + 10_000 * outer_index + 1,
            "training": seed + 10_000 * outer_index + 2,
            "visualization": seed + 10_000 * outer_index + 3,
            "evaluation": seed + 10_000 * outer_index + 4,
        }
        visualization_base = teacher.sample_base(
            visualization_samples,
            cpu_generator(iteration_seeds["visualization"], 301),
        )
        with torch.no_grad():
            old_trajectory = integrate_fixed_rk4(old_student, visualization_base, ode_steps)
            teacher_trajectory = integrate_fixed_rk4(
                teacher.velocity, visualization_base, ode_steps
            )

        cache = generate_source_cache(
            old_student=old_student,
            teacher=teacher,
            cache_size=cache_size,
            ode_steps=ode_steps,
            alpha=float(alpha),
            seed=iteration_seeds["cache"],
        )
        fit_result = fit_student_from_cache(
            old_student=old_student,
            cache=cache,
            updates=inner_updates,
            batch_size=batch_size,
            learning_rate=learning_rate,
            seed=iteration_seeds["training"],
            checkpoint_updates=checkpoint_updates,
            early_stopping_patience=early_stopping_patience,
            max_updates=inner_updates,
        )
        current_student = clone_frozen_model(fit_result.model)
        with torch.no_grad():
            actual_trajectory = integrate_fixed_rk4(
                current_student, visualization_base, ode_steps
            )

        teacher_count = int(cache.teacher_branch.sum())
        records.append(
            OuterIterationRecord(
                outer_iteration=outer_index,
                ideal_accumulated_teacher_weight=1.0
                - (1.0 - float(alpha)) ** (outer_index + 1),
                seeds=iteration_seeds,
                old_trajectory=_detached_trajectory(old_trajectory),
                teacher_trajectory=_detached_trajectory(teacher_trajectory),
                actual_trajectory=_detached_trajectory(actual_trajectory),
                branch_cache_summary={
                    "trajectory_count": float(cache_size),
                    "teacher_count": float(teacher_count),
                    "old_student_count": float(cache_size - teacher_count),
                    "realized_teacher_fraction": teacher_count / cache_size,
                    "requested_teacher_probability": float(alpha),
                },
                train_history=fit_result.history,
                checkpoints=fit_result.checkpoints,
                best_update=fit_result.best_update,
                stopped_early=fit_result.stopped_early,
            )
        )

    return OuterLoopResult(
        final_student=clone_frozen_model(current_student),
        iterations=records,
    )

In [ ]:
import time


_smoke_started_at = time.perf_counter()
_smoke_probe = TEACHER_2D.sample_base(12, cpu_generator(CONFIG.seed, 401))
_smoke_density = TEACHER_2D.density(torch.linspace(0.0, 1.0, 12), _smoke_probe)
_smoke_responsibilities = TEACHER_2D.responsibilities(0.5, _smoke_probe)
_smoke_velocity = TEACHER_2D.velocity(0.5, _smoke_probe)
if _smoke_density.shape != (12,) or (_smoke_density < 0).any():
    raise RuntimeError(
        f"teacher density smoke failed: shape={tuple(_smoke_density.shape)}, "
        f"minimum={float(_smoke_density.min())!r}"
    )
if not torch.allclose(
    _smoke_responsibilities.sum(dim=-1), torch.ones(12), atol=1.0e-5, rtol=1.0e-5
):
    raise RuntimeError("teacher responsibilities smoke failed to sum to one")
if _smoke_velocity.shape != (12, 2) or not torch.isfinite(_smoke_velocity).all():
    raise RuntimeError(
        f"teacher velocity smoke failed: shape={tuple(_smoke_velocity.shape)}"
    )

_smoke_zero_student = VelocityMLP(state_dim=2, condition_dim=0, width=16)
if not torch.equal(_smoke_zero_student(0.5, _smoke_probe), torch.zeros_like(_smoke_probe)):
    raise RuntimeError("zero-initialized VelocityMLP did not produce exact zero velocity")
_smoke_differentiable_base = _smoke_probe[:4].clone().requires_grad_(True)
_, _smoke_diff_states, _ = integrate_fixed_rk4(
    _smoke_zero_student, _smoke_differentiable_base, steps=2
)
_smoke_diff_states[:, -1].sum().backward()
if _smoke_differentiable_base.grad is None or not torch.isfinite(
    _smoke_differentiable_base.grad
).all():
    raise RuntimeError("RK4 smoke failed to preserve gradients to its initial state")

_smoke_outer_kwargs = {
    "teacher": TEACHER_2D,
    "alpha": CONFIG.alpha,
    "outer_iterations": 1,
    "cache_size": 128,
    "inner_updates": 2,
    "batch_size": 32,
    "ode_steps": 4,
    "student_width": 16,
    "learning_rate": CONFIG.learning_rate,
    "seed": CONFIG.seed + 500,
    "visualization_samples": 32,
    "checkpoint_updates": (),
    "early_stopping_patience": 2,
}
_smoke_rng_before = torch.random.get_rng_state().clone()
SMOKE_2D_RESULT = run_mixture_of_density_outer_loop(**_smoke_outer_kwargs)
_smoke_rng_after = torch.random.get_rng_state().clone()
_smoke_repeat_result = run_mixture_of_density_outer_loop(**_smoke_outer_kwargs)
_smoke_rng_after_repeat = torch.random.get_rng_state().clone()
if not (
    torch.equal(_smoke_rng_before, _smoke_rng_after)
    and torch.equal(_smoke_rng_before, _smoke_rng_after_repeat)
):
    raise RuntimeError("outer-loop initialization leaked changes to the global Torch RNG state")

_smoke_record = SMOKE_2D_RESULT.iterations[0]
_smoke_repeat_record = _smoke_repeat_result.iterations[0]
for _parameter_name, _parameter in _smoke_record.checkpoints[0].items():
    if not torch.equal(_parameter, _smoke_repeat_record.checkpoints[0][_parameter_name]):
        raise RuntimeError(
            f"same-seed outer loops initialized differently for parameter {_parameter_name!r}"
        )
for _parameter_name, _parameter in SMOKE_2D_RESULT.final_student.state_dict().items():
    if not torch.equal(
        _parameter, _smoke_repeat_result.final_student.state_dict()[_parameter_name]
    ):
        raise RuntimeError(
            f"same-seed outer loops finished differently for parameter {_parameter_name!r}"
        )
if _smoke_record.train_history != _smoke_repeat_record.train_history:
    raise RuntimeError("same-seed outer loops produced different training histories")
if not torch.equal(
    _smoke_record.actual_trajectory[1], _smoke_repeat_record.actual_trajectory[1]
):
    raise RuntimeError("same-seed outer loops produced different actual trajectories")

_expected_trajectory_shape = (32, 5, 2)
for _trajectory_name, _trajectory in {
    "old": _smoke_record.old_trajectory,
    "teacher": _smoke_record.teacher_trajectory,
    "actual": _smoke_record.actual_trajectory,
}.items():
    _times, _states, _velocities = _trajectory
    if _times.shape != (5,) or _states.shape != _expected_trajectory_shape:
        raise RuntimeError(
            f"{_trajectory_name} trajectory smoke failed: times={tuple(_times.shape)}, "
            f"states={tuple(_states.shape)}"
        )
    if _velocities.shape != _expected_trajectory_shape or not (
        torch.isfinite(_states).all() and torch.isfinite(_velocities).all()
    ):
        raise RuntimeError(
            f"{_trajectory_name} trajectory has invalid velocity shape or non-finite values"
        )
_smoke_teacher_count = int(_smoke_record.branch_cache_summary["teacher_count"])
_smoke_old_count = int(_smoke_record.branch_cache_summary["old_student_count"])
if _smoke_teacher_count <= 0 or _smoke_old_count <= 0:
    raise RuntimeError(
        f"source-cache smoke expected both branches, got teacher={_smoke_teacher_count}, "
        f"old_student={_smoke_old_count}"
    )
_smoke_history_updates = [int(row["update"]) for row in _smoke_record.train_history]
if _smoke_history_updates != [0, 1, 2]:
    raise RuntimeError(
        f"inner fit smoke expected update-0 validation plus two updates, got "
        f"{_smoke_history_updates!r}"
    )
if _smoke_record.best_update not in _smoke_record.checkpoints:
    raise RuntimeError(
        f"best update {_smoke_record.best_update} missing from checkpoints "
        f"{sorted(_smoke_record.checkpoints)!r}"
    )
for _parameter_name, _best_parameter in _smoke_record.checkpoints[
    _smoke_record.best_update
].items():
    if not torch.equal(
        _best_parameter, SMOKE_2D_RESULT.final_student.state_dict()[_parameter_name]
    ):
        raise RuntimeError(
            f"best checkpoint disagrees with selected model for parameter {_parameter_name!r}"
        )
_smoke_train_losses = [
    row["train_loss"] for row in _smoke_record.train_history if "train_loss" in row
]
_smoke_validation_losses = [
    row["validation_loss"] for row in _smoke_record.train_history
]
if not all(math.isfinite(value) for value in _smoke_train_losses + _smoke_validation_losses):
    raise RuntimeError("inner fit smoke produced non-finite numeric history")

SMOKE_2D_SUMMARY = {
    "runtime_seconds": time.perf_counter() - _smoke_started_at,
    "teacher_density_shape": tuple(_smoke_density.shape),
    "trajectory_shape": _expected_trajectory_shape,
    "teacher_branches": _smoke_teacher_count,
    "old_student_branches": _smoke_old_count,
    "history_updates": _smoke_history_updates,
    "best_update": _smoke_record.best_update,
    "checkpoint_updates": sorted(_smoke_record.checkpoints),
    "train_losses": _smoke_train_losses,
    "validation_losses": _smoke_validation_losses,
    "same_seed_reproduced": True,
    "global_rng_preserved": True,
    "finite": True,
}
print(SMOKE_2D_SUMMARY)

## Density diagnostics, metrics, and animations

A plotted endpoint density is a **fixed-grid estimate**, not a training signal. We first count samples with `numpy.histogram2d`, then smooth those counts with a normalized separable Gaussian kernel implemented by two `torch.nn.functional.conv2d` calls. Every frame uses the same bounds, bin edges, bandwidth, sample count, and random seeds. The estimator divides by the **total** sample count, so its integral over the plotting window preserves observed in-window mass instead of renormalizing each distribution to one. Reported `outside_mass` includes both samples outside the window and smoothing mass lost at its boundary. An optional window-normalized array is returned only for explicitly labeled shape-only displays; the residual and training animation use mass-preserving densities.

KDE-like pictures are bandwidth- and grid-dependent. They can merge nearby modes, hide rare modes, create apparent bridges, and obscure tail error. Sliced Wasserstein-2 is likewise a finite-direction projection diagnostic, nearest-mode occupancy depends on the assignment radius, and ODE step-doubling measures numerical discretization rather than distributional fit. For that reason the notebook reports several complementary metrics and always labels the fitted student as **actual / not exact**.

The plug-in responsibility

\[
\widehat\gamma_{k,1}(x)=\frac{\alpha\widehat\rho_{T,1}(x)}{(1-\alpha)\widehat\rho_{S_k,1}(x)+\alpha\widehat\rho_{T,1}(x)}
\]

is shown only as a **matched-KDE endpoint visualization**. Both old and teacher densities use the same sample count, seed policy, grid, bandwidth, and finite-window treatment. It is not an oracle and is never passed to `fit_student_from_cache`, used as a label, or used as an input feature.

The current target samples are deterministic **Bernoulli Monte Carlo** draws: each endpoint independently chooses the old or teacher branch with probability `(1-alpha, alpha)`. They approximate the theoretical current mixture but are not an exact finite realization of its mass. Figures therefore say **empirical current target**, and target-fit mode occupancy is computed from those same empirical samples. Teacher-resemblance metrics remain separately named.

### Three superficially similar but different constructions

- **Correct branch supervision:** choose old or teacher once per complete trajectory, then retain that branch for all times.
- **Wrong control 1 — constant arithmetic velocity average:** evaluate `(1-alpha) * v_old(t, x) + alpha * v_teacher(t, x)`. This ignores density responsibilities and generally transports the wrong marginal.
- **Wrong control 2 — per-step branch redraw/switching:** redraw the old/teacher branch at every ODE step. This creates a switched stochastic numerical process, not samples from the trajectory-level mixture.

A separate tempting construction, **A0 teacher-on-old-state**, evaluates `v_teacher(t, X_old(t))` while states still come from the old trajectory. It is neither correct branch supervision (teacher labels require teacher-branch states) nor the density-weighted marginal field. It is discussed separately because it is a state/label mismatch, whereas the two controls above define alternative dynamics that can be integrated and compared.

In [ ]:
def _validated_2d_samples(samples: object, name: str) -> torch.Tensor:
    if isinstance(samples, np.ndarray):
        samples = torch.from_numpy(samples)
    if not isinstance(samples, torch.Tensor):
        raise TypeError(f"expected {name} to be a torch.Tensor or numpy.ndarray, got {type(samples).__name__}")
    if samples.device.type != "cpu" or samples.ndim != 2 or samples.shape[1] != 2 or samples.shape[0] < 2:
        raise ValueError(
            f"expected CPU {name} shape (sample_count>=2, 2), got device={samples.device}, "
            f"shape={tuple(samples.shape)}"
        )
    samples = samples.detach().to(dtype=torch.float32)
    if not torch.isfinite(samples).all():
        raise ValueError(f"expected finite {name}, got non-finite values")
    return samples


def estimate_endpoint_density(
    samples: object,
    grid_limit: float,
    grid_bins: int,
    bandwidth: float,
) -> Dict[str, np.ndarray]:
    """Estimate a normalized endpoint density on one fixed square grid."""
    samples_tensor = _validated_2d_samples(samples, "density samples")
    if not math.isfinite(grid_limit) or grid_limit <= 0:
        raise ValueError(f"expected finite positive grid_limit, got {grid_limit!r}")
    if isinstance(grid_bins, bool) or not isinstance(grid_bins, int) or grid_bins < 8:
        raise ValueError(f"expected integer grid_bins>=8, got {grid_bins!r}")
    if not math.isfinite(bandwidth) or bandwidth <= 0:
        raise ValueError(f"expected finite positive bandwidth, got {bandwidth!r}")

    edges = np.linspace(-grid_limit, grid_limit, grid_bins + 1, dtype=np.float64)
    sample_array = samples_tensor.numpy().astype(np.float64, copy=False)
    counts, _, _ = np.histogram2d(sample_array[:, 1], sample_array[:, 0], bins=(edges, edges))
    in_bounds_count = float(counts.sum())
    if not math.isfinite(in_bounds_count) or in_bounds_count <= 0:
        raise ValueError(
            f"density grid contains zero finite mass for bounds=[{-grid_limit}, {grid_limit}] "
            f"and sample_count={samples_tensor.shape[0]}"
        )
    bin_width = float(edges[1] - edges[0])
    sigma_bins = bandwidth / bin_width
    radius = max(1, int(math.ceil(4.0 * sigma_bins)))
    offsets = torch.arange(-radius, radius + 1, dtype=torch.float32)
    kernel = torch.exp(-0.5 * (offsets / sigma_bins).square())
    kernel = kernel / kernel.sum()
    if not torch.isfinite(kernel).all() or not torch.isclose(kernel.sum(), torch.tensor(1.0), atol=1.0e-6):
        raise ValueError(
            f"Gaussian kernel normalization failed for bandwidth={bandwidth}, bin_width={bin_width}"
        )
    histogram = torch.from_numpy(counts).to(dtype=torch.float32)[None, None]
    horizontal = F.conv2d(histogram, kernel.reshape(1, 1, 1, -1), padding=(0, radius))
    smoothed = F.conv2d(horizontal, kernel.reshape(1, 1, -1, 1), padding=(radius, 0))[0, 0]
    density = smoothed / (samples_tensor.shape[0] * bin_width * bin_width)
    window_mass = float(density.sum() * bin_width * bin_width)
    in_bounds_fraction = in_bounds_count / samples_tensor.shape[0]
    if window_mass > in_bounds_fraction:
        if window_mass - in_bounds_fraction > 1.0e-5:
            raise ValueError(
                f"smoothed grid mass exceeds histogram in-bounds mass: "
                f"window_mass={window_mass}, in_bounds_fraction={in_bounds_fraction}"
            )
        density = density * (in_bounds_fraction / window_mass)
        window_mass = in_bounds_fraction
    sample_outside_mass = max(0.0, 1.0 - in_bounds_fraction)
    boundary_smoothing_loss = max(0.0, in_bounds_fraction - window_mass)
    outside_mass = max(0.0, 1.0 - window_mass)
    if density.shape != (grid_bins, grid_bins) or not torch.isfinite(density).all():
        raise ValueError(
            f"expected finite density shape ({grid_bins}, {grid_bins}), got {tuple(density.shape)}"
        )
    if not 0.0 < window_mass <= in_bounds_fraction + 1.0e-6:
        raise ValueError(
            f"expected preserved window mass in (0, in_bounds_fraction], got "
            f"window_mass={window_mass}, in_bounds_fraction={in_bounds_fraction}"
        )
    window_normalized_density = density / window_mass
    normalized_mass = float(window_normalized_density.sum() * bin_width * bin_width)
    if not math.isclose(normalized_mass, 1.0, rel_tol=1.0e-5, abs_tol=1.0e-5):
        raise ValueError(f"window-normalized display density mass must be 1, got {normalized_mass!r}")
    centers = 0.5 * (edges[:-1] + edges[1:])
    return {
        "density": density.numpy(),
        "window_normalized_density": window_normalized_density.numpy(),
        "centers": centers,
        "edges": edges,
        "in_bounds_fraction": np.asarray(in_bounds_fraction),
        "sample_outside_mass": np.asarray(sample_outside_mass),
        "boundary_smoothing_loss": np.asarray(boundary_smoothing_loss),
        "window_mass": np.asarray(window_mass),
        "outside_mass": np.asarray(outside_mass),
    }


def sample_empirical_endpoint_mixture(
    old_endpoint_samples: object,
    teacher: TeacherGaussianCrown,
    teacher_weight: float,
    sample_count: int,
    seed: int,
) -> Tuple[torch.Tensor, float]:
    """Draw a deterministic Bernoulli Monte Carlo endpoint mixture."""
    old_samples = _validated_2d_samples(old_endpoint_samples, "old endpoint samples")
    if not isinstance(teacher, TeacherGaussianCrown):
        raise TypeError(f"expected TeacherGaussianCrown, got {type(teacher).__name__}")
    if not math.isfinite(teacher_weight) or not 0.0 <= teacher_weight <= 1.0:
        raise ValueError(f"expected finite teacher_weight in [0, 1], got {teacher_weight!r}")
    if isinstance(sample_count, bool) or not isinstance(sample_count, int) or sample_count < 2:
        raise ValueError(f"expected integer sample_count>=2, got {sample_count!r}")
    if isinstance(seed, bool) or not isinstance(seed, int):
        raise TypeError(f"expected integer mixture seed, got {type(seed).__name__}: {seed!r}")
    teacher_branch = torch.rand(sample_count, generator=cpu_generator(seed, 601)) < teacher_weight
    teacher_count = int(teacher_branch.sum())
    old_count = sample_count - teacher_count
    old_indices = torch.randint(
        old_samples.shape[0], (old_count,), generator=cpu_generator(seed, 602)
    )
    mixed = torch.empty(sample_count, 2, dtype=torch.float32)
    mixed[~teacher_branch] = old_samples[old_indices]
    if teacher_count:
        mixed[teacher_branch] = teacher.sample_endpoint(
            teacher_count, cpu_generator(seed, 603)
        )
    return mixed, teacher_count / sample_count


def sample_ideal_recursion_endpoint(
    initial_endpoint_samples: object,
    teacher: TeacherGaussianCrown,
    alpha: float,
    outer_iteration: int,
    sample_count: int,
    seed: int,
) -> Tuple[torch.Tensor, float]:
    if isinstance(outer_iteration, bool) or not isinstance(outer_iteration, int) or outer_iteration < 1:
        raise ValueError(f"expected outer_iteration>=1, got {outer_iteration!r}")
    ideal_teacher_weight = 1.0 - (1.0 - float(alpha)) ** outer_iteration
    return sample_empirical_endpoint_mixture(
        initial_endpoint_samples,
        teacher,
        ideal_teacher_weight,
        sample_count,
        seed,
    )


def fixed_projection_directions(projection_count: int) -> torch.Tensor:
    if isinstance(projection_count, bool) or not isinstance(projection_count, int) or projection_count < 2:
        raise ValueError(f"expected integer projection_count>=2, got {projection_count!r}")
    angles = (torch.arange(projection_count, dtype=torch.float32) + 0.5) * math.pi / projection_count
    return torch.stack((torch.cos(angles), torch.sin(angles)), dim=-1)


def sliced_wasserstein_2(
    samples_a: object,
    samples_b: object,
    directions: torch.Tensor,
) -> float:
    samples_a = _validated_2d_samples(samples_a, "samples_a")
    samples_b = _validated_2d_samples(samples_b, "samples_b")
    if samples_a.shape[0] != samples_b.shape[0]:
        raise ValueError(
            f"expected equal sample counts for exact sliced-W2 sorting, got {samples_a.shape[0]} and {samples_b.shape[0]}"
        )
    directions = _validated_2d_samples(directions, "projection directions")
    direction_norms = torch.linalg.vector_norm(directions, dim=-1)
    if not torch.allclose(direction_norms, torch.ones_like(direction_norms), atol=1.0e-5, rtol=1.0e-5):
        raise ValueError(f"expected unit projection directions, got norms={direction_norms.tolist()!r}")
    projections_a = torch.sort(samples_a @ directions.T, dim=0).values
    projections_b = torch.sort(samples_b @ directions.T, dim=0).values
    return float(torch.sqrt((projections_a - projections_b).square().mean()))


def _mode_occupancy(
    samples: object,
    teacher: TeacherGaussianCrown,
    assignment_radius: float,
) -> Tuple[torch.Tensor, float]:
    samples_tensor = _validated_2d_samples(samples, "mode occupancy samples")
    standardized = torch.linalg.vector_norm(
        (samples_tensor[:, None, :] - teacher.means[None, :, :]) / teacher.widths[None, :, None],
        dim=-1,
    )
    nearest_distance, nearest_mode = standardized.min(dim=1)
    assigned = nearest_distance <= assignment_radius
    counts = torch.bincount(nearest_mode[assigned], minlength=teacher.num_components).float()
    return counts / samples_tensor.shape[0], float((~assigned).float().mean())


def mode_occupancy_metrics(
    actual_samples: object,
    target_samples: object,
    teacher: TeacherGaussianCrown,
    assignment_radius: float = 3.0,
    coverage_fraction: float = 0.25,
) -> Dict[str, object]:
    if not math.isfinite(assignment_radius) or assignment_radius <= 0:
        raise ValueError(f"expected finite positive assignment_radius, got {assignment_radius!r}")
    if not math.isfinite(coverage_fraction) or not 0.0 < coverage_fraction <= 1.0:
        raise ValueError(f"expected coverage_fraction in (0, 1], got {coverage_fraction!r}")
    actual_masses, actual_background = _mode_occupancy(
        actual_samples, teacher, assignment_radius
    )
    target_masses, target_background = _mode_occupancy(
        target_samples, teacher, assignment_radius
    )
    target_present = target_masses > 0
    target_covered = target_present & (actual_masses >= coverage_fraction * target_masses)
    positive_target_masses = target_masses[target_present]
    rare_threshold = (
        positive_target_masses.median() if positive_target_masses.numel() else torch.tensor(0.0)
    )
    rare_target = target_present & (target_masses <= rare_threshold)
    teacher_covered = actual_masses >= coverage_fraction * teacher.weights
    rare_teacher = teacher.weights <= teacher.weights.median()
    target_mode_count = int(target_present.sum())
    rare_target_count = int(rare_target.sum())
    return {
        "actual_mode_masses": actual_masses.tolist(),
        "target_mode_masses": target_masses.tolist(),
        "target_mode_mass_l1": float(torch.abs(actual_masses - target_masses).sum()),
        "target_mode_coverage": (
            float(target_covered.sum() / target_mode_count) if target_mode_count else 1.0
        ),
        "target_rare_mode_recall": (
            float(target_covered[rare_target].float().mean()) if rare_target_count else 1.0
        ),
        "actual_background_mass": actual_background,
        "target_background_mass": target_background,
        "background_mass_abs_error": abs(actual_background - target_background),
        "teacher_resemblance_mode_mass_l1": float(
            torch.abs(actual_masses - teacher.weights).sum()
        ),
        "teacher_resemblance_coverage": float(teacher_covered.float().mean()),
        "teacher_resemblance_rare_mode_recall": float(
            teacher_covered[rare_teacher].float().mean()
        ),
    }


def endpoint_metric_bundle(
    actual_samples: object,
    target_samples: object,
    teacher_samples: object,
    teacher: TeacherGaussianCrown,
    directions: torch.Tensor,
) -> Dict[str, object]:
    return {
        "sliced_w2_to_empirical_target": sliced_wasserstein_2(
            actual_samples, target_samples, directions
        ),
        "teacher_resemblance_sliced_w2": sliced_wasserstein_2(
            actual_samples, teacher_samples, directions
        ),
        **mode_occupancy_metrics(actual_samples, target_samples, teacher),
    }


def ode_step_doubling_error(
    velocity_model: object,
    initial_samples: object,
    coarse_steps: int,
) -> float:
    initial_samples = _validated_2d_samples(initial_samples, "ODE step-doubling initial samples")
    if isinstance(coarse_steps, bool) or not isinstance(coarse_steps, int) or coarse_steps <= 0:
        raise ValueError(f"expected positive integer coarse_steps, got {coarse_steps!r}")
    with torch.no_grad():
        coarse_endpoint = integrate_fixed_rk4(velocity_model, initial_samples, coarse_steps)[1][:, -1]
        fine_endpoint = integrate_fixed_rk4(velocity_model, initial_samples, 2 * coarse_steps)[1][:, -1]
    return float(torch.sqrt((coarse_endpoint - fine_endpoint).square().sum(dim=-1).mean()))


def reconstruct_velocity_mlp(state_dict: Dict[str, torch.Tensor]) -> VelocityMLP:
    """Reconstruct a VelocityMLP from a visualization-only state_dict snapshot."""
    if not isinstance(state_dict, dict) or not state_dict:
        raise TypeError(f"expected non-empty state_dict, got {type(state_dict).__name__}")
    required = ("network.0.weight", "network.0.bias", "network.4.weight", "network.4.bias")
    missing = [name for name in required if name not in state_dict]
    if missing:
        raise ValueError(f"state_dict is missing VelocityMLP parameters: {missing!r}")
    first_weight = state_dict["network.0.weight"]
    last_weight = state_dict["network.4.weight"]
    if first_weight.ndim != 2 or last_weight.ndim != 2:
        raise ValueError(
            f"expected rank-2 first/last weights, got {tuple(first_weight.shape)} and {tuple(last_weight.shape)}"
        )
    state_dim = int(last_weight.shape[0])
    width = int(first_weight.shape[0])
    condition_dim = int(first_weight.shape[1]) - state_dim - 3
    if state_dim <= 0 or width <= 0 or condition_dim < 0 or last_weight.shape[1] != width:
        raise ValueError(
            f"incompatible VelocityMLP snapshot shapes: first={tuple(first_weight.shape)}, last={tuple(last_weight.shape)}"
        )
    model = VelocityMLP(state_dim=state_dim, condition_dim=condition_dim, width=width)
    model.load_state_dict(state_dict, strict=True)
    return clone_frozen_model(model)


def evaluate_outer_checkpoints(
    outer_result: OuterLoopResult,
    teacher: TeacherGaussianCrown,
    config: DemoConfig,
) -> List[Dict[str, object]]:
    """Evaluate every retained inner/outer checkpoint with fixed seeds and directions."""
    if not isinstance(outer_result, OuterLoopResult) or not outer_result.iterations:
        raise ValueError("expected a non-empty OuterLoopResult")
    metric_count = config.metric_samples
    directions = fixed_projection_directions(config.sw_projections)
    metric_base = teacher.sample_base(metric_count, cpu_generator(config.seed, 701))
    teacher_samples = teacher.sample_endpoint(metric_count, cpu_generator(config.seed, 702))
    initial_model = reconstruct_velocity_mlp(outer_result.iterations[0].checkpoints[0])
    with torch.no_grad():
        initial_endpoint = integrate_fixed_rk4(initial_model, metric_base, config.ode_steps)[1][:, -1]
    evaluations: List[Dict[str, object]] = []
    for record in outer_result.iterations:
        old_model = reconstruct_velocity_mlp(record.checkpoints[0])
        with torch.no_grad():
            old_endpoint = integrate_fixed_rk4(old_model, metric_base, config.ode_steps)[1][:, -1]
        target_samples, realized_target_weight = sample_empirical_endpoint_mixture(
            old_endpoint,
            teacher,
            config.alpha,
            metric_count,
            record.seeds["evaluation"] + 710,
        )
        ideal_samples, realized_ideal_weight = sample_ideal_recursion_endpoint(
            initial_endpoint,
            teacher,
            config.alpha,
            record.outer_iteration + 1,
            metric_count,
            record.seeds["evaluation"] + 711,
        )
        for update in sorted(record.checkpoints):
            model = reconstruct_velocity_mlp(record.checkpoints[update])
            with torch.no_grad():
                trajectory = integrate_fixed_rk4(model, metric_base, config.ode_steps)
            endpoint = trajectory[1][:, -1]
            metrics = endpoint_metric_bundle(endpoint, target_samples, teacher_samples, teacher, directions)
            metrics["sliced_w2_to_empirical_ideal_recursion"] = sliced_wasserstein_2(
                endpoint, ideal_samples, directions
            )
            metrics["ode_step_doubling_error"] = ode_step_doubling_error(
                model, metric_base[: min(256, metric_count)], config.ode_steps
            )
            evaluations.append(
                {
                    "outer_iteration": record.outer_iteration,
                    "inner_update": update,
                    "is_selected_best": update == record.best_update,
                    "realized_target_teacher_weight": realized_target_weight,
                    "realized_ideal_teacher_weight": realized_ideal_weight,
                    "metrics": metrics,
                    "endpoint_samples": endpoint,
                    "target_samples": target_samples,
                    "ideal_samples": ideal_samples,
                    "trajectory": trajectory,
                }
            )
    return evaluations

In [ ]:
def inspect_gif_timing(path: Path) -> Dict[str, object]:
    """Read effective GIF frame count and per-frame timing from the saved file."""
    if not isinstance(path, Path) or not path.is_file():
        raise ValueError(f"expected an existing GIF path, got {path!r}")
    with Image.open(path) as image:
        frame_count = int(getattr(image, "n_frames", 1))
        durations_seconds = []
        timing_available = True
        for frame_index in range(frame_count):
            image.seek(frame_index)
            duration_ms = image.info.get("duration")
            if not isinstance(duration_ms, (int, float)) or duration_ms <= 0:
                timing_available = False
                durations_seconds = []
                break
            durations_seconds.append(float(duration_ms) / 1000.0)
    return {
        "frame_count": frame_count,
        "timing_available": timing_available,
        "durations_seconds": durations_seconds,
        "total_duration_seconds": (
            float(sum(durations_seconds)) if durations_seconds else None
        ),
        "mean_frame_duration_seconds": (
            float(np.mean(durations_seconds)) if durations_seconds else None
        ),
    }


def validated_save_frames(
    frames: Sequence[np.ndarray], stem: str, fps: int
) -> Dict[str, Path]:
    if not frames:
        raise ValueError(f"expected at least one frame for {stem!r}")
    reference_shape = frames[0].shape
    raw_frame_bytes = 0
    for frame_index, frame in enumerate(frames):
        if not isinstance(frame, np.ndarray) or frame.dtype != np.uint8:
            received_dtype = getattr(frame, "dtype", None)
            raise TypeError(
                f"expected uint8 numpy frame {frame_index} for {stem!r}, got "
                f"type={type(frame).__name__}, dtype={received_dtype}"
            )
        if frame.ndim != 3 or frame.shape[2] != 3 or frame.shape != reference_shape:
            raise ValueError(
                f"expected every frame shape {reference_shape} with RGB channels, got "
                f"frame {frame_index} shape={frame.shape}"
            )
        if frame.shape[0] % 16 or frame.shape[1] % 16:
            raise ValueError(
                f"expected macroblock-aligned frame dimensions, got frame {frame_index} shape={frame.shape}"
            )
        raw_frame_bytes += frame.nbytes
    if raw_frame_bytes > 60 * 1024**2:
        raise ValueError(
            f"retained raw frames for {stem!r} exceed 60 MiB: {raw_frame_bytes / 1024**2:.2f} MiB"
        )
    paths = save_frames(frames, stem, fps)
    for format_name, path in paths.items():
        if not path.is_file() or path.stat().st_size <= 0:
            raise RuntimeError(
                f"{format_name.upper()} export for {stem!r} did not create a non-empty file: {path}"
            )
    gif_timing = inspect_gif_timing(paths["gif"])
    if gif_timing["frame_count"] != len(frames):
        raise RuntimeError(
            f"GIF frame count mismatch for {stem!r}: expected {len(frames)}, got {gif_timing['frame_count']}"
        )
    expected_duration = 1.0 / fps
    if len(frames) > 1 and (
        gif_timing["mean_frame_duration_seconds"] is None
        or abs(gif_timing["mean_frame_duration_seconds"] - expected_duration) > 0.02
    ):
        raise RuntimeError(
            f"GIF timing mismatch for {stem!r}: expected about {expected_duration:.6f}s, "
            f"got {gif_timing['mean_frame_duration_seconds']!r}s"
        )
    return paths


def _density_extent(config: DemoConfig) -> Tuple[float, float, float, float]:
    return (-config.grid_limit, config.grid_limit, -config.grid_limit, config.grid_limit)


def _checkpoint_label(evaluation: Dict[str, object]) -> str:
    return f"outer {int(evaluation['outer_iteration']) + 1}, inner {int(evaluation['inner_update'])}"


def build_outer_training_frames(
    outer_result: OuterLoopResult,
    evaluations: Sequence[Dict[str, object]],
    teacher: TeacherGaussianCrown,
    config: DemoConfig,
) -> List[np.ndarray]:
    """Build fixed-axis 2x3 frames for all retained inner checkpoints."""
    if not evaluations:
        raise ValueError("expected at least one checkpoint evaluation")
    metric_base = teacher.sample_base(config.metric_samples, cpu_generator(config.seed, 701))
    teacher_samples = teacher.sample_endpoint(
        config.metric_samples, cpu_generator(config.seed, 702)
    )
    teacher_density = estimate_endpoint_density(
        teacher_samples, config.grid_limit, config.grid_bins, config.kde_bandwidth
    )
    source_densities: Dict[int, Dict[str, np.ndarray]] = {}
    density_vmax = float(teacher_density["density"].max())
    for record in outer_result.iterations:
        old_model = reconstruct_velocity_mlp(record.checkpoints[0])
        with torch.no_grad():
            old_endpoint = integrate_fixed_rk4(old_model, metric_base, config.ode_steps)[1][:, -1]
        source_density = estimate_endpoint_density(
            old_endpoint, config.grid_limit, config.grid_bins, config.kde_bandwidth
        )
        source_densities[record.outer_iteration] = source_density
        density_vmax = max(density_vmax, float(source_density["density"].max()))

    prepared: List[Dict[str, object]] = []
    residual_limit = 0.0
    for evaluation in evaluations:
        actual_density = estimate_endpoint_density(
            evaluation["endpoint_samples"], config.grid_limit, config.grid_bins, config.kde_bandwidth
        )
        target_density = estimate_endpoint_density(
            evaluation["target_samples"], config.grid_limit, config.grid_bins, config.kde_bandwidth
        )
        residual = actual_density["density"] - target_density["density"]
        residual_limit = max(residual_limit, float(np.max(np.abs(residual))))
        density_vmax = max(
            density_vmax,
            float(actual_density["density"].max()),
            float(target_density["density"].max()),
        )
        prepared.append(
            {
                "evaluation": evaluation,
                "actual_density": actual_density,
                "target_density": target_density,
                "residual": residual,
            }
        )
    if not math.isfinite(residual_limit) or residual_limit <= 0:
        residual_limit = 1.0e-6
    if not math.isfinite(density_vmax) or density_vmax <= 0:
        raise ValueError(f"expected finite positive fixed density vmax, got {density_vmax!r}")
    contour_levels = np.geomspace(max(density_vmax / 128.0, 1.0e-8), density_vmax, 6)
    centers = teacher_density["centers"]
    bin_width = float(teacher_density["edges"][1] - teacher_density["edges"][0])
    extent = _density_extent(config)
    x_values = np.arange(len(prepared))
    target_curve = np.asarray(
        [
            item["evaluation"]["metrics"]["sliced_w2_to_empirical_target"]
            for item in prepared
        ],
        dtype=float,
    )
    teacher_curve = np.asarray(
        [
            item["evaluation"]["metrics"]["teacher_resemblance_sliced_w2"]
            for item in prepared
        ],
        dtype=float,
    )
    ideal_curve = np.asarray(
        [
            item["evaluation"]["metrics"]["sliced_w2_to_empirical_ideal_recursion"]
            for item in prepared
        ],
        dtype=float,
    )
    curve_max = max(
        float(target_curve.max()),
        float(teacher_curve.max()),
        float(ideal_curve.max()),
        1.0e-6,
    )
    global_mode_ymax = max(
        0.05,
        1.15
        * max(
            max(item["evaluation"]["metrics"]["actual_mode_masses"])
            for item in prepared
        ),
        1.15
        * max(
            max(item["evaluation"]["metrics"]["target_mode_masses"])
            for item in prepared
        ),
    )
    outer_indices = np.asarray(
        [int(item["evaluation"]["outer_iteration"]) for item in prepared], dtype=int
    )
    boundary_starts = [
        index
        for index in range(1, len(prepared))
        if outer_indices[index] != outer_indices[index - 1]
    ]
    segment_starts = [0, *boundary_starts]
    segment_ends = [*boundary_starts, len(prepared)]
    outer_tick_positions = [
        0.5 * (start + end - 1) for start, end in zip(segment_starts, segment_ends)
    ]
    outer_tick_labels = [f"q{outer_indices[start]}" for start in segment_starts]
    frames: List[np.ndarray] = []
    for frame_index, item in enumerate(prepared):
        evaluation = item["evaluation"]
        outer_index = int(evaluation["outer_iteration"])
        source_estimate = source_densities[outer_index]
        source_density = source_estimate["density"]
        target_estimate = item["target_density"]
        target_density = target_estimate["density"]
        actual_estimate = item["actual_density"]
        actual_density = actual_estimate["density"]
        trajectory_states = evaluation["trajectory"][1]
        actual_mode_masses = np.asarray(
            evaluation["metrics"]["actual_mode_masses"], dtype=float
        )
        target_mode_masses = np.asarray(
            evaluation["metrics"]["target_mode_masses"], dtype=float
        )
        residual_integral = float(item["residual"].sum() * bin_width**2)

        # Keep all retained checkpoint frames below the validated 60 MiB raw-memory budget.
        fig, axes = plt.subplots(
            2, 3, figsize=(12, 7.2), dpi=40, constrained_layout=True
        )
        axes[0, 0].contour(
            centers,
            centers,
            source_density,
            levels=contour_levels,
            colors="tab:blue",
            linewidths=1.0,
        )
        axes[0, 0].contour(
            centers,
            centers,
            target_density,
            levels=contour_levels,
            colors="tab:orange",
            linewidths=1.0,
        )
        axes[0, 0].contour(
            centers,
            centers,
            teacher_density["density"],
            levels=contour_levels,
            colors="tab:purple",
            linestyles="dashed",
            linewidths=0.7,
        )
        axes[0, 0].set_title(
            "fixed levels: source blue / empirical target orange / teacher purple",
            fontsize=8,
        )

        axes[0, 1].imshow(
            actual_density,
            extent=extent,
            origin="lower",
            cmap="magma",
            vmin=0.0,
            vmax=density_vmax,
        )
        axes[0, 1].set_title(
            f"actual fitted student — not exact\noutside mass={float(actual_estimate['outside_mass']):.3f}",
            fontsize=9,
        )

        residual_image = axes[0, 2].imshow(
            item["residual"],
            extent=extent,
            origin="lower",
            cmap="coolwarm",
            vmin=-residual_limit,
            vmax=residual_limit,
        )
        axes[0, 2].set_title(
            f"actual − empirical target; ∫window={residual_integral:+.3f}\n"
            f"outside actual/target={float(actual_estimate['outside_mass']):.3f}/"
            f"{float(target_estimate['outside_mass']):.3f}",
            fontsize=8,
        )
        fig.colorbar(residual_image, ax=axes[0, 2], fraction=0.046)

        path_count = min(32, trajectory_states.shape[0])
        for path_index in range(path_count):
            path = trajectory_states[path_index].numpy()
            axes[1, 0].plot(path[:, 0], path[:, 1], color="tab:green", alpha=0.22, linewidth=0.7)
        endpoint_array = evaluation["endpoint_samples"].numpy()
        axes[1, 0].scatter(endpoint_array[:512, 0], endpoint_array[:512, 1], s=5, alpha=0.35)
        axes[1, 0].set_title("fixed-seed fitted-student paths/endpoints")

        curve_specs = (
            (target_curve, "SW2 empirical target", "tab:blue"),
            (teacher_curve, "teacher resemblance SW2", "tab:orange"),
            (ideal_curve, "SW2 empirical ideal recursion", "tab:green"),
        )
        for curve_values, curve_label, curve_color in curve_specs:
            label_pending = True
            for segment_start, segment_end in zip(segment_starts, segment_ends):
                visible_end = min(segment_end, frame_index + 1)
                if visible_end <= segment_start:
                    continue
                axes[1, 1].plot(
                    x_values[segment_start:visible_end],
                    curve_values[segment_start:visible_end],
                    color=curve_color,
                    label=curve_label if label_pending else None,
                )
                label_pending = False
        for boundary_start in boundary_starts:
            axes[1, 1].axvline(
                boundary_start - 0.5,
                color="0.35",
                linestyle=":",
                linewidth=0.8,
            )
        axes[1, 1].set_xlim(-0.5, max(0.5, len(prepared) - 0.5))
        axes[1, 1].set_ylim(0.0, 1.05 * curve_max)
        axes[1, 1].set_xticks(outer_tick_positions, outer_tick_labels)
        axes[1, 1].set_xlabel("outer target segment")
        axes[1, 1].set_title(
            "segmented SW2 curves; dotted lines mark qₖ changes", fontsize=8
        )
        axes[1, 1].legend(fontsize=6)

        mode_indices = np.arange(teacher.num_components)
        axes[1, 2].bar(
            mode_indices - 0.18, target_mode_masses, width=0.36, label="empirical target"
        )
        axes[1, 2].bar(
            mode_indices + 0.18, actual_mode_masses, width=0.36, label="actual"
        )
        axes[1, 2].set_ylim(0.0, global_mode_ymax)
        axes[1, 2].set_title(
            f"target-fit occupancy L1={evaluation['metrics']['target_mode_mass_l1']:.3f}\n"
            f"background actual/target={evaluation['metrics']['actual_background_mass']:.3f}/"
            f"{evaluation['metrics']['target_background_mass']:.3f}",
            fontsize=8,
        )
        axes[1, 2].legend(fontsize=7)

        for axis in (axes[0, 0], axes[0, 1], axes[0, 2], axes[1, 0]):
            axis.set_xlim(-config.grid_limit, config.grid_limit)
            axis.set_ylim(-config.grid_limit, config.grid_limit)
            axis.set_aspect("equal")
        best_suffix = " | selected best" if evaluation["is_selected_best"] else ""
        fig.suptitle(
            f"Outer iteration {outer_index + 1}/{len(outer_result.iterations)} | "
            f"inner update {int(evaluation['inner_update'])}{best_suffix}\n"
            f"empirical target is fixed within outer {outer_index + 1} and changes at the next outer boundary",
            fontsize=10,
        )
        frame = figure_to_rgb(fig)
        plt.close(fig)
        frames.append(frame)
    return frames


def build_generation_trajectory_frames(
    outer_result: OuterLoopResult,
    teacher: TeacherGaussianCrown,
    config: DemoConfig,
    particle_count: int = 512,
) -> List[np.ndarray]:
    """Animate frozen old, analytic teacher, and final student over ODE time."""
    if isinstance(particle_count, bool) or not isinstance(particle_count, int) or particle_count < 16:
        raise ValueError(f"expected integer particle_count>=16, got {particle_count!r}")
    initial_old = reconstruct_velocity_mlp(outer_result.iterations[0].checkpoints[0])
    base = teacher.sample_base(particle_count, cpu_generator(config.seed, 801))
    with torch.no_grad():
        old_trajectory = integrate_fixed_rk4(initial_old, base, config.ode_steps)
        teacher_trajectory = integrate_fixed_rk4(teacher.velocity, base, config.ode_steps)
        final_trajectory = integrate_fixed_rk4(outer_result.final_student, base, config.ode_steps)
    times = old_trajectory[0]
    frames: List[np.ndarray] = []
    colors = ("tab:blue", "tab:orange", "tab:green")
    titles = ("frozen initial old", "analytic teacher", "final fitted student — not exact")
    trajectories = (old_trajectory[1], teacher_trajectory[1], final_trajectory[1])
    for time_index, time_value in enumerate(times):
        # The complete ODE-time sequence must fit the same 60 MiB raw-frame budget.
        fig, axes = plt.subplots(
            1, 3, figsize=(12, 7.2), dpi=40, constrained_layout=True
        )
        for axis, states, color, title in zip(axes, trajectories, colors, titles):
            path_count = min(24, particle_count)
            for path_index in range(path_count):
                path = states[path_index, : time_index + 1].numpy()
                axis.plot(path[:, 0], path[:, 1], color=color, alpha=0.25, linewidth=0.7)
            particles = states[:, time_index].numpy()
            axis.scatter(particles[:, 0], particles[:, 1], s=7, alpha=0.45, color=color)
            axis.set_xlim(-config.grid_limit, config.grid_limit)
            axis.set_ylim(-config.grid_limit, config.grid_limit)
            axis.set_aspect("equal")
            axis.set_title(title)
        fig.suptitle(f"Generation trajectories at ODE time t={float(time_value):.3f}")
        frame = figure_to_rgb(fig)
        plt.close(fig)
        frames.append(frame)
    return frames


def integrate_per_step_branch_redraw(
    old_model: object,
    teacher: TeacherGaussianCrown,
    initial_samples: object,
    steps: int,
    alpha: float,
    seed: int,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Integrate the intentionally wrong control that redraws a branch each RK4 step."""
    state = _validated_2d_samples(initial_samples, "branch-redraw initial samples").clone()
    if isinstance(steps, bool) or not isinstance(steps, int) or steps <= 0:
        raise ValueError(f"expected positive integer steps, got {steps!r}")
    if not math.isfinite(alpha) or not 0.0 <= alpha <= 1.0:
        raise ValueError(f"expected finite alpha in [0, 1], got {alpha!r}")
    times = torch.linspace(0.0, 1.0, steps + 1)
    step_size = 1.0 / steps
    stored = [state.clone()]
    generator = cpu_generator(seed, 802)
    for step_index in range(steps):
        teacher_branch = torch.rand(state.shape[0], generator=generator) < alpha

        def switched_velocity(time_value: object, state_value: torch.Tensor) -> torch.Tensor:
            velocity = torch.empty_like(state_value)
            if teacher_branch.any():
                velocity[teacher_branch] = teacher.velocity(time_value, state_value[teacher_branch])
            if (~teacher_branch).any():
                velocity[~teacher_branch] = old_model(time_value, state_value[~teacher_branch])
            return velocity

        time_value = times[step_index]
        midpoint = time_value + 0.5 * step_size
        k1 = switched_velocity(time_value, state)
        k2 = switched_velocity(midpoint, state + 0.5 * step_size * k1)
        k3 = switched_velocity(midpoint, state + 0.5 * step_size * k2)
        k4 = switched_velocity(time_value + step_size, state + step_size * k3)
        state = state + (step_size / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
        if not torch.isfinite(state).all():
            raise ValueError(f"per-step branch redraw produced non-finite state at step {step_index + 1}")
        stored.append(state.clone())
    return times, torch.stack(stored, dim=1)


def evaluate_wrong_controls(
    outer_result: OuterLoopResult,
    teacher: TeacherGaussianCrown,
    config: DemoConfig,
) -> Dict[str, object]:
    """Evaluate and plot two visibly labeled incorrect dynamics controls."""
    old_model = reconstruct_velocity_mlp(outer_result.iterations[0].checkpoints[0])
    sample_count = config.metric_samples
    base = teacher.sample_base(sample_count, cpu_generator(config.seed, 811))

    def arithmetic_velocity(time_value: object, state: torch.Tensor) -> torch.Tensor:
        return (1.0 - config.alpha) * old_model(time_value, state) + config.alpha * teacher.velocity(time_value, state)

    with torch.no_grad():
        old_endpoint = integrate_fixed_rk4(old_model, base, config.ode_steps)[1][:, -1]
        arithmetic_trajectory = integrate_fixed_rk4(arithmetic_velocity, base, config.ode_steps)
        redraw_times, redraw_states = integrate_per_step_branch_redraw(
            old_model, teacher, base, config.ode_steps, config.alpha, config.seed + 812
        )
    target_samples, _ = sample_empirical_endpoint_mixture(
        old_endpoint, teacher, config.alpha, sample_count, config.seed + 813
    )
    teacher_samples = teacher.sample_endpoint(sample_count, cpu_generator(config.seed, 814))
    directions = fixed_projection_directions(config.sw_projections)
    endpoints = {
        "wrong constant arithmetic velocity average": arithmetic_trajectory[1][:, -1],
        "wrong per-step branch redraw/switching": redraw_states[:, -1],
    }
    metrics = {
        name: endpoint_metric_bundle(samples, target_samples, teacher_samples, teacher, directions)
        for name, samples in endpoints.items()
    }
    metrics["wrong constant arithmetic velocity average"]["ode_step_doubling_error"] = ode_step_doubling_error(
        arithmetic_velocity, base[:256], config.ode_steps
    )
    metrics["wrong per-step branch redraw/switching"]["ode_step_doubling_error"] = None

    plot_sets = [("empirical current target", target_samples), *endpoints.items()]
    density_estimates = [
        estimate_endpoint_density(
            samples, config.grid_limit, config.grid_bins, config.kde_bandwidth
        )
        for _, samples in plot_sets
    ]
    density_vmax = max(float(estimate["density"].max()) for estimate in density_estimates)
    if not math.isfinite(density_vmax) or density_vmax <= 0:
        raise ValueError(
            f"expected finite positive common wrong-control density vmax, got {density_vmax!r}"
        )
    contour_levels = np.geomspace(max(density_vmax / 128.0, 1.0e-8), density_vmax, 6)
    centers = density_estimates[0]["centers"]
    fig, axes = plt.subplots(
        1, 3, figsize=(12, 7.2), dpi=80, constrained_layout=True
    )
    density_image = None
    for axis, (title, _), estimate in zip(axes, plot_sets, density_estimates):
        density_image = axis.imshow(
            estimate["density"],
            extent=_density_extent(config),
            origin="lower",
            cmap="magma",
            vmin=0.0,
            vmax=density_vmax,
        )
        axis.contour(
            centers,
            centers,
            estimate["density"],
            levels=contour_levels,
            colors="white",
            linewidths=0.45,
            alpha=0.7,
        )
        axis.set_xlim(-config.grid_limit, config.grid_limit)
        axis.set_ylim(-config.grid_limit, config.grid_limit)
        axis.set_aspect("equal")
        axis.set_title(
            f"{title}\noutside mass={float(estimate['outside_mass']):.3f}", fontsize=9
        )
    if density_image is None:
        raise RuntimeError("wrong-control density comparison produced no image")
    fig.colorbar(
        density_image,
        ax=axes.tolist(),
        fraction=0.025,
        label="common mass-preserving density scale",
    )
    comparison_path = OUTPUT_DIR / "wrong_controls_comparison.png"
    fig.savefig(comparison_path)
    plt.close(fig)
    if not comparison_path.is_file() or comparison_path.stat().st_size <= 0:
        raise RuntimeError(f"wrong-control comparison figure was not written: {comparison_path}")
    return {
        "endpoint_samples": endpoints,
        "metrics": metrics,
        "comparison_path": comparison_path,
        "redraw_times": redraw_times,
    }


def plot_endpoint_matched_kde_responsibility(
    outer_result: OuterLoopResult,
    teacher: TeacherGaussianCrown,
    config: DemoConfig,
) -> Path:
    """Plot visualization-only endpoint gamma from matched old/teacher KDE estimates."""
    old_model = reconstruct_velocity_mlp(outer_result.iterations[0].checkpoints[0])
    base = teacher.sample_base(config.metric_samples, cpu_generator(config.seed, 821))
    teacher_samples = teacher.sample_endpoint(
        config.metric_samples, cpu_generator(config.seed, 822)
    )
    with torch.no_grad():
        old_endpoint = integrate_fixed_rk4(old_model, base, config.ode_steps)[1][:, -1]
    old_grid = estimate_endpoint_density(
        old_endpoint, config.grid_limit, config.grid_bins, config.kde_bandwidth
    )
    teacher_grid = estimate_endpoint_density(
        teacher_samples, config.grid_limit, config.grid_bins, config.kde_bandwidth
    )
    denominator = (
        (1.0 - config.alpha) * old_grid["density"]
        + config.alpha * teacher_grid["density"]
    )
    gamma = np.divide(
        config.alpha * teacher_grid["density"],
        denominator,
        out=np.zeros_like(teacher_grid["density"]),
        where=denominator > 0,
    )
    if not np.isfinite(gamma).all() or gamma.min() < 0 or gamma.max() > 1.0 + 1.0e-6:
        raise ValueError(
            f"matched-KDE responsibility must be finite in [0, 1], got "
            f"range=({gamma.min()}, {gamma.max()})"
        )
    fig, axis = plt.subplots(figsize=(7.2, 7.2), dpi=80, constrained_layout=True)
    image = axis.imshow(
        gamma, extent=_density_extent(config), origin="lower", cmap="viridis", vmin=0, vmax=1
    )
    axis.set_title(
        "Matched-KDE plug-in endpoint responsibility γ\n"
        "equal samples/grid/bandwidth/window; visualization only"
    )
    axis.set_aspect("equal")
    fig.colorbar(image, ax=axis, label="plug-in teacher responsibility")
    path = OUTPUT_DIR / "endpoint_matched_kde_responsibility.png"
    fig.savefig(path)
    plt.close(fig)
    if not path.is_file() or path.stat().st_size <= 0:
        raise RuntimeError(f"matched-KDE responsibility figure was not written: {path}")
    return path


def _json_ready(value: object) -> object:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, torch.Tensor):
        if value.numel() == 1:
            return float(value)
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        if value.ndim == 0:
            return float(value)
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): _json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(item) for item in value]
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    raise TypeError(f"cannot serialize value of type {type(value).__name__} to metrics JSON")


def select_final_best_evaluation(
    outer_result: OuterLoopResult,
    evaluations: Sequence[Dict[str, object]],
    teacher: TeacherGaussianCrown,
    config: DemoConfig,
) -> Dict[str, object]:
    """Locate and validate the final outer iteration's recorded best evaluation."""
    if not isinstance(outer_result, OuterLoopResult) or not outer_result.iterations:
        raise ValueError("expected a non-empty OuterLoopResult for final-best selection")
    if not isinstance(evaluations, Sequence) or not evaluations:
        raise ValueError("expected at least one checkpoint evaluation for final-best selection")
    if not isinstance(teacher, TeacherGaussianCrown):
        raise TypeError(
            f"expected TeacherGaussianCrown for final-best selection, got {type(teacher).__name__}"
        )
    if not isinstance(config, DemoConfig):
        raise TypeError(
            f"expected DemoConfig for final-best selection, got {type(config).__name__}"
        )

    final_record = outer_result.iterations[-1]
    candidates = [
        evaluation
        for evaluation in evaluations
        if int(evaluation["outer_iteration"]) == final_record.outer_iteration
        and int(evaluation["inner_update"]) == final_record.best_update
    ]
    if len(candidates) != 1:
        raise RuntimeError(
            "expected exactly one final-best checkpoint evaluation, got "
            f"count={len(candidates)}, final_outer_iteration={final_record.outer_iteration}, "
            f"recorded_best_update={final_record.best_update}"
        )
    selected = candidates[0]
    if selected.get("is_selected_best") is not True:
        raise RuntimeError(
            "final-best evaluation is not marked selected: "
            f"outer_iteration={final_record.outer_iteration}, "
            f"inner_update={final_record.best_update}, "
            f"is_selected_best={selected.get('is_selected_best')!r}"
        )
    if final_record.best_update not in final_record.checkpoints:
        raise RuntimeError(
            "final outer record is missing its best checkpoint: "
            f"best_update={final_record.best_update}, "
            f"available_updates={sorted(final_record.checkpoints)!r}"
        )

    final_state = outer_result.final_student.state_dict()
    best_state = final_record.checkpoints[final_record.best_update]
    if final_state.keys() != best_state.keys():
        raise RuntimeError(
            "final student and recorded best checkpoint have different parameter keys: "
            f"final_keys={sorted(final_state)!r}, best_keys={sorted(best_state)!r}"
        )
    mismatched_parameters = [
        name for name in final_state if not torch.equal(final_state[name].cpu(), best_state[name].cpu())
    ]
    if mismatched_parameters:
        raise RuntimeError(
            "final student parameters do not match the recorded best checkpoint: "
            f"mismatched_parameters={mismatched_parameters!r}, "
            f"best_update={final_record.best_update}"
        )

    metric_base = teacher.sample_base(
        config.metric_samples, cpu_generator(config.seed, 701)
    )
    with torch.no_grad():
        final_trajectory = integrate_fixed_rk4(
            outer_result.final_student, metric_base, config.ode_steps
        )
    final_endpoint = final_trajectory[1][:, -1]
    selected_endpoint = selected.get("endpoint_samples")
    selected_trajectory = selected.get("trajectory")
    if not isinstance(selected_endpoint, torch.Tensor):
        raise TypeError(
            "expected final-best endpoint_samples to be torch.Tensor, got "
            f"{type(selected_endpoint).__name__}"
        )
    if not isinstance(selected_trajectory, tuple) or len(selected_trajectory) != 3:
        raise TypeError(
            "expected final-best trajectory tuple (times, states, velocities), got "
            f"{type(selected_trajectory).__name__}"
        )
    if not torch.equal(selected_endpoint, selected_trajectory[1][:, -1]):
        raise RuntimeError(
            "final-best endpoint_samples disagree with the retained evaluation trajectory: "
            f"endpoint_shape={tuple(selected_endpoint.shape)}, "
            f"trajectory_shape={tuple(selected_trajectory[1].shape)}"
        )
    if not torch.equal(final_endpoint, selected_endpoint):
        raise RuntimeError(
            "final student outputs do not match the final-best checkpoint evaluation: "
            f"final_endpoint_shape={tuple(final_endpoint.shape)}, "
            f"selected_endpoint_shape={tuple(selected_endpoint.shape)}, "
            f"best_update={final_record.best_update}"
        )
    return selected

In [ ]:
# Normal FAST_MODE execution runs the complete 2D training and exports. The
# environment-controlled test mode keeps only the deterministic smoke paths below.
RUN_2D_TRAINING = not MIXTURE_DENSITY_TEST_MODE

TRAINING_2D_ARTIFACTS = None
if RUN_2D_TRAINING:
    if DEVICE.type != "cpu":
        raise RuntimeError(f"this demo is CPU-only, got DEVICE={DEVICE}")
    if not CONFIG.fast_mode:
        raise RuntimeError(
            "full mode is never auto-run; set RUN_2D_TRAINING=False or explicitly call the outer loop yourself"
        )
    training_started_at = time.perf_counter()
    OUTER_2D_RESULT = run_mixture_of_density_outer_loop(
        teacher=TEACHER_2D,
        alpha=CONFIG.alpha,
        outer_iterations=CONFIG.outer_iterations,
        cache_size=CONFIG.cache_size,
        inner_updates=CONFIG.inner_updates,
        batch_size=CONFIG.batch_size,
        ode_steps=CONFIG.ode_steps,
        student_width=CONFIG.student_width,
        learning_rate=CONFIG.learning_rate,
        seed=CONFIG.seed + 1_000,
        visualization_samples=CONFIG.visualization_samples,
        checkpoint_updates=CONFIG.checkpoint_updates,
        early_stopping_patience=max(50, CONFIG.inner_updates // 3),
    )
    CHECKPOINT_2D_EVALUATIONS = evaluate_outer_checkpoints(
        OUTER_2D_RESULT, TEACHER_2D, CONFIG
    )
    FINAL_BEST_2D_EVALUATION = select_final_best_evaluation(
        OUTER_2D_RESULT, CHECKPOINT_2D_EVALUATIONS, TEACHER_2D, CONFIG
    )
    # Retain every checkpoint frame while making the selected final-best checkpoint
    # the animation's concluding frame rather than the largest inner-update number.
    ANIMATION_2D_EVALUATIONS = [
        evaluation
        for evaluation in CHECKPOINT_2D_EVALUATIONS
        if evaluation is not FINAL_BEST_2D_EVALUATION
    ] + [FINAL_BEST_2D_EVALUATION]
    if len(ANIMATION_2D_EVALUATIONS) != len(CHECKPOINT_2D_EVALUATIONS):
        raise RuntimeError(
            "final-best animation ordering changed the checkpoint count: "
            f"ordered={len(ANIMATION_2D_EVALUATIONS)}, "
            f"evaluated={len(CHECKPOINT_2D_EVALUATIONS)}"
        )
    OUTER_TRAINING_FRAMES = build_outer_training_frames(
        OUTER_2D_RESULT, ANIMATION_2D_EVALUATIONS, TEACHER_2D, CONFIG
    )
    OUTER_TRAINING_PATHS = validated_save_frames(
        OUTER_TRAINING_FRAMES, "outer_training", CONFIG.export_fps
    )
    GENERATION_TRAJECTORY_FRAMES = build_generation_trajectory_frames(
        OUTER_2D_RESULT, TEACHER_2D, CONFIG
    )
    GENERATION_TRAJECTORY_PATHS = validated_save_frames(
        GENERATION_TRAJECTORY_FRAMES, "generation_trajectories", CONFIG.export_fps
    )
    WRONG_CONTROL_RESULTS = evaluate_wrong_controls(
        OUTER_2D_RESULT, TEACHER_2D, CONFIG
    )
    MATCHED_KDE_RESPONSIBILITY_PATH = plot_endpoint_matched_kde_responsibility(
        OUTER_2D_RESULT, TEACHER_2D, CONFIG
    )

    checkpoint_metrics = [
        {
            "outer_iteration": int(evaluation["outer_iteration"]),
            "inner_update": int(evaluation["inner_update"]),
            "is_selected_best": bool(evaluation["is_selected_best"]),
            "realized_target_teacher_weight": evaluation["realized_target_teacher_weight"],
            "realized_ideal_teacher_weight": evaluation["realized_ideal_teacher_weight"],
            "metrics": evaluation["metrics"],
        }
        for evaluation in CHECKPOINT_2D_EVALUATIONS
    ]
    training_runtime = time.perf_counter() - training_started_at
    final_actual_density = estimate_endpoint_density(
        FINAL_BEST_2D_EVALUATION["endpoint_samples"],
        CONFIG.grid_limit,
        CONFIG.grid_bins,
        CONFIG.kde_bandwidth,
    )
    final_target_density = estimate_endpoint_density(
        FINAL_BEST_2D_EVALUATION["target_samples"],
        CONFIG.grid_limit,
        CONFIG.grid_bins,
        CONFIG.kde_bandwidth,
    )
    final_density_window = {
        "actual_window_mass": final_actual_density["window_mass"],
        "actual_outside_mass": final_actual_density["outside_mass"],
        "actual_sample_outside_mass": final_actual_density["sample_outside_mass"],
        "target_window_mass": final_target_density["window_mass"],
        "target_outside_mass": final_target_density["outside_mass"],
        "target_sample_outside_mass": final_target_density["sample_outside_mass"],
        "residual_window_integral": float(
            final_actual_density["window_mass"] - final_target_density["window_mass"]
        ),
    }
    metrics_payload = {
        "config": asdict(CONFIG),
        "runtime_seconds": training_runtime,
        "checkpoint_evaluations": checkpoint_metrics,
        "final_best_evaluation": {
            "outer_iteration": int(FINAL_BEST_2D_EVALUATION["outer_iteration"]),
            "inner_update": int(FINAL_BEST_2D_EVALUATION["inner_update"]),
            "metrics": FINAL_BEST_2D_EVALUATION["metrics"],
        },
        "final_density_window": final_density_window,
        "wrong_controls": WRONG_CONTROL_RESULTS["metrics"],
        "artifacts": {
            "outer_training": OUTER_TRAINING_PATHS,
            "generation_trajectories": GENERATION_TRAJECTORY_PATHS,
            "wrong_controls_comparison": WRONG_CONTROL_RESULTS["comparison_path"],
            "endpoint_matched_kde_responsibility": MATCHED_KDE_RESPONSIBILITY_PATH,
        },
    }
    METRICS_2D_PATH = OUTPUT_DIR / "metrics_2d.json"
    with METRICS_2D_PATH.open("w", encoding="utf-8") as handle:
        json.dump(_json_ready(metrics_payload), handle, indent=2)
    if not METRICS_2D_PATH.is_file() or METRICS_2D_PATH.stat().st_size <= 0:
        raise RuntimeError(f"metrics JSON was not written: {METRICS_2D_PATH}")

    TRAINING_2D_ARTIFACTS = {
        "runtime_seconds": training_runtime,
        "outer_frame_count": len(OUTER_TRAINING_FRAMES),
        "outer_frame_shape": OUTER_TRAINING_FRAMES[0].shape,
        "outer_raw_frame_mib": sum(frame.nbytes for frame in OUTER_TRAINING_FRAMES) / 1024**2,
        "outer_gif_timing": inspect_gif_timing(OUTER_TRAINING_PATHS["gif"]),
        "generation_frame_count": len(GENERATION_TRAJECTORY_FRAMES),
        "generation_frame_shape": GENERATION_TRAJECTORY_FRAMES[0].shape,
        "generation_raw_frame_mib": sum(
            frame.nbytes for frame in GENERATION_TRAJECTORY_FRAMES
        )
        / 1024**2,
        "generation_gif_timing": inspect_gif_timing(
            GENERATION_TRAJECTORY_PATHS["gif"]
        ),
        "checkpoint_count": len(CHECKPOINT_2D_EVALUATIONS),
        "final_best_outer_iteration": int(FINAL_BEST_2D_EVALUATION["outer_iteration"]),
        "final_best_update": int(FINAL_BEST_2D_EVALUATION["inner_update"]),
        "final_best_metrics": FINAL_BEST_2D_EVALUATION["metrics"],
        "final_density_window": final_density_window,
        "metrics_path": METRICS_2D_PATH,
        "outer_training_paths": OUTER_TRAINING_PATHS,
        "generation_trajectory_paths": GENERATION_TRAJECTORY_PATHS,
        "wrong_control_path": WRONG_CONTROL_RESULTS["comparison_path"],
        "matched_kde_responsibility_path": MATCHED_KDE_RESPONSIBILITY_PATH,
    }
    print(_json_ready(TRAINING_2D_ARTIFACTS))

In [ ]:
VISUAL_SMOKE = True

VISUAL_SMOKE_SUMMARY = None
if VISUAL_SMOKE:
    visual_smoke_started_at = time.perf_counter()
    smoke_config = DemoConfig(
        fast_mode=True,
        alpha=CONFIG.alpha,
        seed=CONFIG.seed + 2_000,
        outer_iterations=1,
        inner_updates=2,
        cache_size=128,
        batch_size=32,
        ode_steps=4,
        student_width=16,
        learning_rate=CONFIG.learning_rate,
        visualization_samples=32,
        metric_samples=64,
        checkpoint_updates=(0, 2),
        grid_limit=CONFIG.grid_limit,
        grid_bins=32,
        kde_bandwidth=CONFIG.kde_bandwidth,
        sw_projections=16,
        export_fps=2,
    )
    smoke_evaluations = evaluate_outer_checkpoints(
        SMOKE_2D_RESULT, TEACHER_2D, smoke_config
    )
    smoke_evaluation = select_final_best_evaluation(
        SMOKE_2D_RESULT, smoke_evaluations, TEACHER_2D, smoke_config
    )
    smoke_kde = estimate_endpoint_density(
        smoke_evaluation["endpoint_samples"],
        smoke_config.grid_limit,
        smoke_config.grid_bins,
        smoke_config.kde_bandwidth,
    )
    if smoke_kde["density"].shape != (smoke_config.grid_bins, smoke_config.grid_bins):
        raise RuntimeError(
            f"visual smoke KDE shape mismatch: {smoke_kde['density'].shape}"
        )
    smoke_window_mass = float(smoke_kde["window_mass"])
    smoke_outside_mass = float(smoke_kde["outside_mass"])
    if not 0.0 < smoke_window_mass <= 1.0 or not math.isclose(
        smoke_window_mass + smoke_outside_mass, 1.0, abs_tol=1.0e-6
    ):
        raise RuntimeError(
            f"visual smoke KDE mass mismatch: window={smoke_window_mass}, "
            f"outside={smoke_outside_mass}"
        )
    smoke_metrics = smoke_evaluation["metrics"]
    required_smoke_metrics = (
        "sliced_w2_to_empirical_target",
        "teacher_resemblance_sliced_w2",
        "sliced_w2_to_empirical_ideal_recursion",
        "target_mode_mass_l1",
        "target_mode_coverage",
        "target_rare_mode_recall",
        "actual_background_mass",
        "target_background_mass",
        "background_mass_abs_error",
        "teacher_resemblance_mode_mass_l1",
        "ode_step_doubling_error",
    )
    for metric_name in required_smoke_metrics:
        metric_value = smoke_metrics[metric_name]
        if not isinstance(metric_value, (int, float)) or not math.isfinite(float(metric_value)):
            raise RuntimeError(
                f"visual smoke metric {metric_name!r} must be finite numeric, got {metric_value!r}"
            )
    smoke_frames = build_outer_training_frames(
        SMOKE_2D_RESULT, [smoke_evaluation], TEACHER_2D, smoke_config
    )
    if len(smoke_frames) != 1:
        raise RuntimeError(f"visual smoke expected one frame, got {len(smoke_frames)}")
    smoke_paths = validated_save_frames(
        smoke_frames, "visual_smoke", smoke_config.export_fps
    )
    VISUAL_SMOKE_SUMMARY = {
        "runtime_seconds": time.perf_counter() - visual_smoke_started_at,
        "selected_outer_iteration": int(smoke_evaluation["outer_iteration"]),
        "selected_best_update": int(smoke_evaluation["inner_update"]),
        "kde_shape": smoke_kde["density"].shape,
        "window_mass": smoke_window_mass,
        "outside_mass": smoke_outside_mass,
        "sample_outside_mass": float(smoke_kde["sample_outside_mass"]),
        "in_bounds_fraction": float(smoke_kde["in_bounds_fraction"]),
        "metrics": smoke_metrics,
        "frame_count": len(smoke_frames),
        "frame_shape": smoke_frames[0].shape,
        "frame_dtype": str(smoke_frames[0].dtype),
        "raw_frame_mib": sum(frame.nbytes for frame in smoke_frames) / 1024**2,
        "gif_timing": inspect_gif_timing(smoke_paths["gif"]),
        "paths": smoke_paths,
    }
    print(_json_ready(VISUAL_SMOKE_SUMMARY))

## Conditional 8×8 image analogy

For image generation, distributional complexity is best understood relative to the conditioning and representation. A prompt or class can admit several **conditional modes** (different valid layouts or appearances), including **rare semantics** that carry little probability but matter qualitatively. Difficulty also comes from the geometry of high-density regions in latent or perceptual space, from structure at several spatial scales, and from high-frequency detail such as thin edges and alternating textures. These are not the same as teacher parameter count or raw entropy: a parameter-free multimodal law can be hard for a small student, while a high-entropy unimodal law can be easy.

This 8×8 experiment is an **analogy**, not a reproduction of a DiT, a VAE, text conditioning, or a production image pipeline. It makes the relevant distributional features visible with three one-hot conditions and four modes per condition:

- condition 0: bars at different locations;
- condition 1: two diagonals, a ring, or four corners;
- condition 2: checkerboard and stripe textures.

Each condition uses unequal probabilities and includes a 3% rare mode. Pixels are mapped from `[0, 1]` to `[-1, 1]` before defining a narrow Gaussian around every template. The analytic teacher uses the same linear path as the 2D section. In 64 dimensions its responsibilities are evaluated in log space with `logsumexp`/`softmax`, avoiding underflow from directly multiplying small Gaussian densities.

In [ ]:
def build_tiny_image_templates() -> Tuple[torch.Tensor, Tuple[Tuple[str, ...], ...], torch.Tensor]:
    """Build deterministic conditional 8x8 templates and nonuniform mode weights."""
    templates = torch.zeros(3, 4, 8, 8, dtype=torch.float32, device=DEVICE)
    names = (
        ("left vertical bar", "right vertical bar", "top horizontal bar", "bottom horizontal bar"),
        ("main diagonal", "anti-diagonal", "ring", "corners"),
        ("checkerboard", "inverse checkerboard", "vertical stripes", "horizontal stripes"),
    )

    templates[0, 0, :, 1:3] = 1.0
    templates[0, 1, :, 5:7] = 1.0
    templates[0, 2, 1:3, :] = 1.0
    templates[0, 3, 5:7, :] = 1.0
    pixel_indices = torch.arange(8)
    templates[1, 0, pixel_indices, pixel_indices] = 1.0
    templates[1, 1, pixel_indices, 7 - pixel_indices] = 1.0
    templates[1, 2, 1:7, 1] = 1.0
    templates[1, 2, 1:7, 6] = 1.0
    templates[1, 2, 1, 1:7] = 1.0
    templates[1, 2, 6, 1:7] = 1.0
    templates[1, 3, :2, :2] = 1.0
    templates[1, 3, :2, -2:] = 1.0
    templates[1, 3, -2:, :2] = 1.0
    templates[1, 3, -2:, -2:] = 1.0
    row_grid = torch.arange(8)[:, None]
    column_grid = torch.arange(8)[None, :]
    templates[2, 0] = ((row_grid + column_grid) % 2).float()
    templates[2, 1] = 1.0 - templates[2, 0]
    templates[2, 2, :, ::2] = 1.0
    templates[2, 3, ::2, :] = 1.0
    weights = torch.tensor(
        ((0.50, 0.30, 0.17, 0.03), (0.45, 0.32, 0.20, 0.03), (0.55, 0.25, 0.17, 0.03)),
        dtype=torch.float32,
        device=DEVICE,
    )
    if templates.shape != (3, 4, 8, 8) or not torch.isfinite(templates).all():
        raise RuntimeError(f"expected finite template shape (3, 4, 8, 8), got {tuple(templates.shape)}")
    if not ((templates == 0.0) | (templates == 1.0)).all():
        raise RuntimeError("tiny image templates must contain only deterministic binary pixels")
    if weights.shape != (3, 4) or not torch.allclose(weights.sum(dim=1), torch.ones(3)):
        raise RuntimeError(f"expected normalized conditional weights shape (3, 4), got {weights.tolist()!r}")
    if not torch.equal(weights[:, -1], torch.full((3,), 0.03)):
        raise RuntimeError(f"expected final mode to be rare with weight 0.03, got {weights[:, -1].tolist()!r}")
    return templates, names, weights


TINY_TEMPLATES_01, TINY_TEMPLATE_NAMES, TINY_MODE_WEIGHTS = build_tiny_image_templates()
TINY_TEMPLATE_MEANS = 2.0 * TINY_TEMPLATES_01.reshape(3, 4, 64) - 1.0

fig, axes = plt.subplots(3, 4, figsize=(8, 6), constrained_layout=True)
for condition_index in range(3):
    for mode_index in range(4):
        axes[condition_index, mode_index].imshow(
            TINY_TEMPLATES_01[condition_index, mode_index], cmap="gray", vmin=0.0, vmax=1.0
        )
        axes[condition_index, mode_index].set_title(
            f"c{condition_index} m{mode_index}: {TINY_MODE_WEIGHTS[condition_index, mode_index]:.0%}\n"
            f"{TINY_TEMPLATE_NAMES[condition_index][mode_index]}",
            fontsize=8,
        )
        axes[condition_index, mode_index].axis("off")
fig.suptitle("Validated deterministic 8×8 conditional templates (fixed [0, 1] scale)")
if not MIXTURE_DENSITY_TEST_MODE and "agg" not in plt.get_backend().lower():
    plt.show()
plt.close(fig)

In [ ]:
class ConditionalImageGaussianTeacher:
    """Represent a conditional Gaussian mixture around 8x8 image templates."""

    def __init__(
        self,
        means: torch.Tensor,
        weights: torch.Tensor,
        widths: Sequence[float] = (0.10, 0.12, 0.08, 0.10),
    ) -> None:
        if not isinstance(means, torch.Tensor) or means.device.type != "cpu":
            raise TypeError(f"expected CPU means tensor, got {type(means).__name__}")
        if means.shape != (3, 4, 64) or not torch.isfinite(means).all():
            raise ValueError(f"expected finite means shape (3, 4, 64), got {tuple(means.shape)}")
        if not isinstance(weights, torch.Tensor) or weights.device.type != "cpu":
            raise TypeError(f"expected CPU weights tensor, got {type(weights).__name__}")
        if weights.shape != (3, 4) or not torch.isfinite(weights).all() or (weights <= 0).any():
            raise ValueError(f"expected finite positive weights shape (3, 4), got {weights.tolist()!r}")
        normalized_weights = weights.float() / weights.float().sum(dim=1, keepdim=True)
        widths_tensor = torch.as_tensor(widths, dtype=torch.float32, device=DEVICE)
        if widths_tensor.shape != (4,) or not torch.isfinite(widths_tensor).all() or (widths_tensor <= 0).any():
            raise ValueError(f"expected four finite positive widths, got {widths_tensor.tolist()!r}")
        self.means = means.float().clone()
        self.weights = normalized_weights
        self.widths = widths_tensor
        self.state_dim = 64
        self.condition_dim = 3
        self.mode_count = 4

    def _validated_condition(self, condition: torch.Tensor, batch_size: int) -> torch.Tensor:
        if not isinstance(condition, torch.Tensor):
            raise TypeError(f"expected condition torch.Tensor, got {type(condition).__name__}")
        if condition.device.type != "cpu" or condition.shape != (batch_size, self.condition_dim):
            raise ValueError(
                f"expected CPU one-hot condition shape ({batch_size}, {self.condition_dim}), got "
                f"device={condition.device}, shape={tuple(condition.shape)}"
            )
        condition = condition.to(dtype=torch.float32)
        if not torch.isfinite(condition).all():
            raise ValueError("expected finite one-hot conditions")
        if not (((condition == 0.0) | (condition == 1.0)).all() and (condition.sum(dim=1) == 1.0).all()):
            raise ValueError(f"expected strict one-hot conditions, got rows={condition.tolist()!r}")
        return condition

    def _validated_state_time(
        self, time_value: object, state: torch.Tensor, condition: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        if not isinstance(state, torch.Tensor):
            raise TypeError(f"expected state torch.Tensor, got {type(state).__name__}")
        if state.device.type != "cpu" or state.ndim != 2 or state.shape[1] != self.state_dim:
            raise ValueError(
                f"expected CPU state shape (batch, {self.state_dim}), got device={state.device}, "
                f"shape={tuple(state.shape)}"
            )
        state = state.to(dtype=torch.float32)
        if not torch.isfinite(state).all():
            raise ValueError("expected finite image states")
        condition = self._validated_condition(condition, state.shape[0])
        time_tensor = torch.as_tensor(time_value, dtype=torch.float32, device=DEVICE)
        if time_tensor.ndim == 0:
            time_tensor = time_tensor.expand(state.shape[0]).reshape(-1, 1)
        elif time_tensor.ndim == 1 and time_tensor.shape[0] == state.shape[0]:
            time_tensor = time_tensor[:, None]
        elif time_tensor.shape != (state.shape[0], 1):
            raise ValueError(
                f"expected scalar time or shape ({state.shape[0]},) / ({state.shape[0]}, 1), "
                f"got {tuple(time_tensor.shape)}"
            )
        if not torch.isfinite(time_tensor).all() or (time_tensor < 0).any() or (time_tensor > 1).any():
            raise ValueError("expected finite times in [0, 1]")
        return time_tensor, state, condition

    def _component_terms(
        self, time_value: object, state: torch.Tensor, condition: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        time_tensor, state, condition = self._validated_state_time(time_value, state, condition)
        conditional_means = torch.einsum("bc,cmd->bmd", condition, self.means)
        conditional_weights = condition @ self.weights
        variances = (1.0 - time_tensor).square() + time_tensor.square() * self.widths.square()[None, :]
        residuals = state[:, None, :] - time_tensor[:, None, :] * conditional_means
        log_density = -0.5 * (
            self.state_dim * (math.log(2.0 * math.pi) + torch.log(variances))
            + residuals.square().sum(dim=-1) / variances
        )
        log_weighted = torch.log(conditional_weights) + log_density
        log_responsibilities = log_weighted - torch.logsumexp(log_weighted, dim=1, keepdim=True)
        if not torch.isfinite(log_responsibilities).all():
            raise ValueError("conditional 64D log-responsibilities contain non-finite values")
        return time_tensor, residuals, conditional_means, variances, log_responsibilities

    def responsibilities(
        self, time_value: object, state: torch.Tensor, condition: torch.Tensor
    ) -> torch.Tensor:
        """Evaluate stable exact conditional responsibilities in 64 dimensions."""
        log_responsibilities = self._component_terms(time_value, state, condition)[-1]
        responsibilities = torch.exp(log_responsibilities)
        if not torch.allclose(responsibilities.sum(dim=1), torch.ones(state.shape[0]), atol=1.0e-5):
            raise ValueError("conditional responsibilities do not sum to one")
        return responsibilities

    def velocity(
        self, time_value: object, state: torch.Tensor, condition: torch.Tensor
    ) -> torch.Tensor:
        """Evaluate the exact responsibility-weighted linear-path velocity."""
        time_tensor, residuals, conditional_means, variances, log_responsibilities = self._component_terms(
            time_value, state, condition
        )
        coefficients = (
            time_tensor * self.widths.square()[None, :] - (1.0 - time_tensor)
        ) / variances
        component_velocities = conditional_means + coefficients[:, :, None] * residuals
        velocity = (torch.exp(log_responsibilities)[:, :, None] * component_velocities).sum(dim=1)
        if velocity.shape != state.shape or not torch.isfinite(velocity).all():
            raise ValueError(
                f"expected finite conditional teacher velocity shape {tuple(state.shape)}, got {tuple(velocity.shape)}"
            )
        return velocity

    def sample_base(self, sample_count: int, generator: torch.Generator) -> torch.Tensor:
        if isinstance(sample_count, bool) or not isinstance(sample_count, int) or sample_count <= 0:
            raise ValueError(f"expected positive integer sample_count, got {sample_count!r}")
        return torch.randn(sample_count, self.state_dim, generator=generator, device=DEVICE)

    def sample_endpoint(
        self, condition: torch.Tensor, generator: torch.Generator
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        condition = self._validated_condition(condition, condition.shape[0])
        condition_indices = condition.argmax(dim=1)
        mode_indices = torch.empty(condition.shape[0], dtype=torch.long, device=DEVICE)
        for condition_index in range(self.condition_dim):
            mask = condition_indices == condition_index
            if mask.any():
                mode_indices[mask] = torch.multinomial(
                    self.weights[condition_index], int(mask.sum()), replacement=True, generator=generator
                )
        noise = torch.randn(condition.shape[0], self.state_dim, generator=generator, device=DEVICE)
        endpoints = self.means[condition_indices, mode_indices] + self.widths[mode_indices, None] * noise
        if not torch.isfinite(endpoints).all():
            raise ValueError("conditional teacher endpoint sampling produced non-finite values")
        return endpoints, mode_indices


TINY_IMAGE_TEACHER = ConditionalImageGaussianTeacher(TINY_TEMPLATE_MEANS, TINY_MODE_WEIGHTS)
_probe_condition = torch.eye(3, dtype=torch.float32).repeat_interleave(4, dim=0)
_probe_state = TINY_IMAGE_TEACHER.sample_base(12, cpu_generator(CONFIG.seed, 900))
_probe_responsibilities = TINY_IMAGE_TEACHER.responsibilities(0.5, _probe_state, _probe_condition)
_probe_velocity = TINY_IMAGE_TEACHER.velocity(0.5, _probe_state, _probe_condition)
if _probe_responsibilities.shape != (12, 4) or _probe_velocity.shape != (12, 64):
    raise RuntimeError(
        f"conditional teacher probe failed: responsibilities={tuple(_probe_responsibilities.shape)}, "
        f"velocity={tuple(_probe_velocity.shape)}"
    )

In [ ]:
@dataclass(frozen=True)
class ConditionalImageSourceCache:
    """Store condition-aware branch-once trajectories and detached labels."""

    times: torch.Tensor
    states: torch.Tensor
    velocities: torch.Tensor
    conditions: torch.Tensor
    teacher_branch: torch.Tensor

    def __post_init__(self) -> None:
        trajectory_count = self.states.shape[0] if self.states.ndim == 3 else -1
        if self.times.ndim != 1 or self.states.ndim != 3 or self.states.shape[2] != 64:
            raise ValueError(
                f"expected times (T,) and states (N,T,64), got {tuple(self.times.shape)}, {tuple(self.states.shape)}"
            )
        if self.velocities.shape != self.states.shape or self.states.shape[1] != self.times.numel():
            raise ValueError(
                f"expected velocity shape {tuple(self.states.shape)} and {self.times.numel()} stored times, "
                f"got {tuple(self.velocities.shape)}"
            )
        if self.conditions.shape != (trajectory_count, 3) or self.teacher_branch.shape != (trajectory_count,):
            raise ValueError(
                f"expected conditions ({trajectory_count}, 3) and branch ({trajectory_count},), got "
                f"{tuple(self.conditions.shape)}, {tuple(self.teacher_branch.shape)}"
            )
        if self.teacher_branch.dtype != torch.bool:
            raise TypeError(f"expected bool teacher_branch, got {self.teacher_branch.dtype}")
        if any(value.device.type != "cpu" for value in (self.times, self.states, self.velocities, self.conditions)):
            raise ValueError("conditional image source cache must remain on CPU")
        if not all(torch.isfinite(value).all() for value in (self.times, self.states, self.velocities, self.conditions)):
            raise ValueError("conditional image source cache contains non-finite values")
        expected_one_hot = (self.conditions.sum(dim=1) == 1.0).all() and (
            (self.conditions == 0.0) | (self.conditions == 1.0)
        ).all()
        if not expected_one_hot:
            raise ValueError("conditional image source cache contains invalid one-hot conditions")


def balanced_tiny_conditions(sample_count: int, seed: int) -> torch.Tensor:
    """Create deterministically shuffled conditions with every condition represented."""
    if isinstance(sample_count, bool) or not isinstance(sample_count, int) or sample_count < 3:
        raise ValueError(f"expected integer sample_count>=3, got {sample_count!r}")
    indices = torch.arange(sample_count, device=DEVICE) % 3
    permutation = torch.randperm(sample_count, generator=cpu_generator(seed, 901))
    conditions = F.one_hot(indices[permutation], num_classes=3).float()
    if (conditions.sum(dim=0) == 0).any():
        raise RuntimeError(f"balanced conditions omitted a condition: counts={conditions.sum(dim=0).tolist()!r}")
    return conditions


def generate_conditional_image_cache(
    old_student: VelocityMLP,
    teacher: ConditionalImageGaussianTeacher,
    cache_size: int,
    ode_steps: int,
    alpha: float,
    seed: int,
) -> ConditionalImageSourceCache:
    """Choose old/teacher once per trajectory and retain that branch at every RK4 time."""
    if not isinstance(old_student, VelocityMLP) or old_student.state_dim != 64 or old_student.condition_dim != 3:
        raise TypeError("expected VelocityMLP(state_dim=64, condition_dim=3) as old_student")
    if not isinstance(teacher, ConditionalImageGaussianTeacher):
        raise TypeError(f"expected ConditionalImageGaussianTeacher, got {type(teacher).__name__}")
    if isinstance(cache_size, bool) or not isinstance(cache_size, int) or cache_size < 6:
        raise ValueError(f"expected integer cache_size>=6, got {cache_size!r}")
    if isinstance(ode_steps, bool) or not isinstance(ode_steps, int) or ode_steps <= 0:
        raise ValueError(f"expected positive integer ode_steps, got {ode_steps!r}")
    if not math.isfinite(float(alpha)) or not 0.0 < float(alpha) < 1.0:
        raise ValueError(f"expected finite alpha in (0, 1), got {alpha!r}")
    frozen_old = clone_frozen_model(old_student)
    conditions = balanced_tiny_conditions(cache_size, seed)
    base = teacher.sample_base(cache_size, cpu_generator(seed, 902))
    teacher_branch = torch.rand(cache_size, generator=cpu_generator(seed, 903)) < float(alpha)
    states = torch.empty(cache_size, ode_steps + 1, 64, dtype=torch.float32, device=DEVICE)
    velocities = torch.empty_like(states)
    times = torch.linspace(0.0, 1.0, ode_steps + 1, device=DEVICE)
    with torch.no_grad():
        old_mask = ~teacher_branch
        if old_mask.any():
            old_times, old_states, old_velocities = integrate_fixed_rk4(
                frozen_old, base[old_mask], ode_steps, conditions[old_mask]
            )
            times = old_times
            states[old_mask] = old_states
            velocities[old_mask] = old_velocities
        if teacher_branch.any():
            teacher_times, teacher_states, teacher_velocities = integrate_fixed_rk4(
                teacher.velocity, base[teacher_branch], ode_steps, conditions[teacher_branch]
            )
            times = teacher_times
            states[teacher_branch] = teacher_states
            velocities[teacher_branch] = teacher_velocities
    return ConditionalImageSourceCache(
        times.detach(), states.detach(), velocities.detach(), conditions.detach(), teacher_branch.detach()
    )


def fit_conditional_image_student(
    old_student: VelocityMLP,
    cache: ConditionalImageSourceCache,
    updates: int,
    batch_size: int,
    learning_rate: float,
    seed: int,
) -> Tuple[VelocityMLP, List[float], bool]:
    """Warm-start the conditional student and fit detached source labels."""
    for name, value in (("updates", updates), ("batch_size", batch_size)):
        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
            raise ValueError(f"expected positive integer {name}, got {value!r}")
    if not math.isfinite(learning_rate) or learning_rate <= 0:
        raise ValueError(f"expected finite positive learning_rate, got {learning_rate!r}")
    student = copy.deepcopy(old_student).to(device=DEVICE, dtype=torch.float32)
    student.train().requires_grad_(True)
    optimizer = torch.optim.Adam(student.parameters(), lr=learning_rate)
    generator = cpu_generator(seed, 904)
    losses: List[float] = []
    finite_gradients = True
    for update_index in range(updates):
        trajectory_indices = torch.randint(cache.states.shape[0], (batch_size,), generator=generator)
        time_indices = torch.randint(cache.times.numel(), (batch_size,), generator=generator)
        states = cache.states[trajectory_indices, time_indices]
        labels = cache.velocities[trajectory_indices, time_indices].detach()
        conditions = cache.conditions[trajectory_indices]
        prediction = student(cache.times[time_indices], states, conditions)
        loss = _event_mean_mse(prediction, labels)
        if not torch.isfinite(loss):
            raise ValueError(f"conditional image fit produced non-finite loss at update {update_index + 1}")
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        gradients = [parameter.grad for parameter in student.parameters() if parameter.grad is not None]
        finite_gradients = finite_gradients and bool(gradients) and all(
            torch.isfinite(gradient).all().item() for gradient in gradients
        )
        if not finite_gradients:
            raise ValueError(f"conditional image fit produced non-finite gradients at update {update_index + 1}")
        optimizer.step()
        losses.append(float(loss.detach()))
    student.eval().requires_grad_(False)
    return student, losses, finite_gradients


@dataclass
class TinyImageOuterRecord:
    outer_iteration: int
    old_student: VelocityMLP
    student: VelocityMLP
    cache: ConditionalImageSourceCache
    actual_endpoints: torch.Tensor
    target_endpoints: torch.Tensor
    conditions: torch.Tensor
    target_teacher_branch: torch.Tensor
    losses: List[float]
    metrics: Dict[str, object]
    frame: np.ndarray


@dataclass
class TinyImageOuterResult:
    final_student: VelocityMLP
    records: List[TinyImageOuterRecord]
    runtime_seconds: float
    gif_path: Optional[Path]

In [ ]:
def tiny_image_mode_occupancy(
    endpoints: torch.Tensor,
    conditions: torch.Tensor,
    template_means: torch.Tensor,
    teacher_widths: torch.Tensor,
    sigma_multiplier: float = 3.0,
) -> Dict[str, torch.Tensor]:
    """Assign only endpoints within a mode-specific per-pixel RMSE threshold."""
    if endpoints.device.type != "cpu" or endpoints.ndim != 2 or endpoints.shape[1] != 64:
        raise ValueError(f"expected CPU endpoints shape (N, 64), got {tuple(endpoints.shape)}")
    if conditions.shape != (endpoints.shape[0], 3):
        raise ValueError(f"expected conditions shape ({endpoints.shape[0]}, 3), got {tuple(conditions.shape)}")
    if template_means.shape != (3, 4, 64) or teacher_widths.shape != (4,):
        raise ValueError(
            f"expected template_means (3, 4, 64) and teacher_widths (4,), got "
            f"{tuple(template_means.shape)} and {tuple(teacher_widths.shape)}"
        )
    if not math.isfinite(sigma_multiplier) or sigma_multiplier <= 0:
        raise ValueError(f"expected finite positive sigma_multiplier, got {sigma_multiplier!r}")
    if not torch.isfinite(endpoints).all():
        raise ValueError("expected finite endpoints for occupancy")
    occupancy = torch.zeros(3, 4, dtype=torch.float32)
    counts = torch.zeros(3, 4, dtype=torch.int64)
    background_mass = torch.zeros(3, dtype=torch.float32)
    condition_counts = torch.zeros(3, dtype=torch.int64)
    condition_indices = conditions.argmax(dim=1)
    for condition_index in range(3):
        mask = condition_indices == condition_index
        sample_count = int(mask.sum())
        if sample_count == 0:
            raise ValueError(f"occupancy requires samples for condition {condition_index}")
        per_pixel_rmse = torch.cdist(
            endpoints[mask].float(), template_means[condition_index].float()
        ) / math.sqrt(64.0)
        nearest_rmse, nearest_modes = per_pixel_rmse.min(dim=1)
        assigned = nearest_rmse <= sigma_multiplier * teacher_widths[nearest_modes]
        assigned_counts = torch.bincount(nearest_modes[assigned], minlength=4)
        counts[condition_index] = assigned_counts
        occupancy[condition_index] = assigned_counts.float() / sample_count
        background_mass[condition_index] = (~assigned).float().mean()
        condition_counts[condition_index] = sample_count
    if not torch.allclose(occupancy.sum(dim=1) + background_mass, torch.ones(3), atol=1.0e-6):
        raise ValueError("mode occupancy and background mass do not sum to one")
    return {
        "occupancy": occupancy,
        "counts": counts,
        "background_mass": background_mass,
        "condition_counts": condition_counts,
    }


def _tiny_frequency_and_diversity(
    endpoints: torch.Tensor, condition_indices: torch.Tensor
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Measure pairwise diversity and non-DC-normalized high-frequency energy."""
    pairwise_diversity = torch.empty(3)
    high_frequency_energy = torch.empty(3)
    frequency = torch.fft.fftfreq(8)
    frequency_y, frequency_x = torch.meshgrid(frequency, frequency, indexing="ij")
    radial_frequency = torch.sqrt(frequency_x.square() + frequency_y.square())
    high_frequency_mask = radial_frequency >= 0.25
    non_dc_mask = radial_frequency > 0.0
    for condition_index in range(3):
        condition_samples = endpoints[condition_indices == condition_index].float()
        if condition_samples.shape[0] < 2:
            raise ValueError(
                f"frequency/diversity metrics require at least two samples for condition {condition_index}"
            )
        pairwise_diversity[condition_index] = torch.pdist(condition_samples).mean()
        spectral_power = torch.fft.fft2(condition_samples.reshape(-1, 8, 8)).abs().square()
        non_dc_power = spectral_power[:, non_dc_mask].sum()
        if not torch.isfinite(non_dc_power) or non_dc_power <= 0:
            raise ValueError(
                f"expected finite positive non-DC FFT power for condition {condition_index}, got {float(non_dc_power)!r}"
            )
        high_frequency_energy[condition_index] = spectral_power[:, high_frequency_mask].sum() / non_dc_power
    return pairwise_diversity, high_frequency_energy


def tiny_image_metrics(
    actual_endpoints: torch.Tensor,
    target_endpoints: torch.Tensor,
    conditions: torch.Tensor,
    sigma_multiplier: float = 3.0,
    coverage_fraction: float = 0.5,
    minimum_resolvable_count: int = 5,
) -> Dict[str, object]:
    """Compute background-aware, target-relative conditional image metrics."""
    if actual_endpoints.shape != target_endpoints.shape:
        raise ValueError(
            f"expected matching actual/target endpoint shapes, got {tuple(actual_endpoints.shape)} and "
            f"{tuple(target_endpoints.shape)}"
        )
    if not math.isfinite(coverage_fraction) or not 0.0 < coverage_fraction <= 1.0:
        raise ValueError(f"expected coverage_fraction in (0, 1], got {coverage_fraction!r}")
    if isinstance(minimum_resolvable_count, bool) or not isinstance(minimum_resolvable_count, int) or minimum_resolvable_count < 2:
        raise ValueError(
            f"expected integer minimum_resolvable_count>=2, got {minimum_resolvable_count!r}"
        )
    actual_assignment = tiny_image_mode_occupancy(
        actual_endpoints, conditions, TINY_TEMPLATE_MEANS, TINY_IMAGE_TEACHER.widths, sigma_multiplier
    )
    target_assignment = tiny_image_mode_occupancy(
        target_endpoints, conditions, TINY_TEMPLATE_MEANS, TINY_IMAGE_TEACHER.widths, sigma_multiplier
    )
    actual_occupancy = actual_assignment["occupancy"]
    target_occupancy = target_assignment["occupancy"]
    actual_counts = actual_assignment["counts"]
    target_counts = target_assignment["counts"]
    occupancy_l1 = torch.abs(actual_occupancy - target_occupancy).sum(dim=1)
    resolvable = target_counts >= minimum_resolvable_count
    required_actual_counts = torch.ceil(coverage_fraction * target_counts.float()).to(torch.int64)
    covered = resolvable & (actual_counts >= required_actual_counts)
    missing_modes = (resolvable & ~covered).sum(dim=1)
    rare_resolvable = resolvable[:, 3]
    rare_recall: List[Optional[bool]] = [
        bool(covered[condition_index, 3]) if bool(rare_resolvable[condition_index]) else None
        for condition_index in range(3)
    ]
    condition_indices = conditions.argmax(dim=1)
    actual_diversity, actual_high_frequency = _tiny_frequency_and_diversity(
        actual_endpoints, condition_indices
    )
    target_diversity, target_high_frequency = _tiny_frequency_and_diversity(
        target_endpoints, condition_indices
    )
    numeric_metrics = (
        actual_occupancy,
        target_occupancy,
        actual_assignment["background_mass"],
        target_assignment["background_mass"],
        occupancy_l1,
        actual_diversity,
        target_diversity,
        actual_high_frequency,
        target_high_frequency,
    )
    if not all(torch.isfinite(metric).all() for metric in numeric_metrics):
        raise ValueError("tiny image metrics contain non-finite numeric values")
    return {
        "assignment_sigma_multiplier": float(sigma_multiplier),
        "assignment_rmse_thresholds": (sigma_multiplier * TINY_IMAGE_TEACHER.widths).tolist(),
        "coverage_fraction_of_target_count": float(coverage_fraction),
        "minimum_resolvable_count": minimum_resolvable_count,
        "actual_occupancy_excluding_background": actual_occupancy.tolist(),
        "target_occupancy_including_old_student_excluding_background": target_occupancy.tolist(),
        "actual_background_mass": actual_assignment["background_mass"].tolist(),
        "target_background_mass": target_assignment["background_mass"].tolist(),
        "background_mass_gap": (
            actual_assignment["background_mass"] - target_assignment["background_mass"]
        ).tolist(),
        "target_relative_occupancy_l1": occupancy_l1.tolist(),
        "target_mode_counts": target_counts.tolist(),
        "actual_mode_counts": actual_counts.tolist(),
        "required_actual_counts_for_coverage": required_actual_counts.tolist(),
        "target_relative_mode_resolvable": resolvable.tolist(),
        "target_relative_mode_covered": [
            [bool(covered[row, column]) if bool(resolvable[row, column]) else None for column in range(4)]
            for row in range(3)
        ],
        "missing_resolvable_modes": missing_modes.tolist(),
        "rare_mode_resolvable": rare_resolvable.tolist(),
        "rare_mode_recall": rare_recall,
        "actual_pairwise_diversity": actual_diversity.tolist(),
        "target_pairwise_diversity": target_diversity.tolist(),
        "target_relative_pairwise_diversity_gap": (actual_diversity - target_diversity).tolist(),
        "actual_fft_high_frequency_energy_non_dc": actual_high_frequency.tolist(),
        "target_fft_high_frequency_energy_non_dc": target_high_frequency.tolist(),
        "target_relative_fft_high_frequency_gap": (
            actual_high_frequency - target_high_frequency
        ).tolist(),
    }


def _tiny_image_montage(endpoints: torch.Tensor, tiles: int = 16) -> np.ndarray:
    if endpoints.ndim != 2 or endpoints.shape[1] != 64 or endpoints.shape[0] < tiles:
        raise ValueError(f"expected at least {tiles} endpoints shape (N, 64), got {tuple(endpoints.shape)}")
    images = ((endpoints[:tiles].reshape(tiles, 8, 8) + 1.0) / 2.0).clamp(0.0, 1.0)
    side = int(math.sqrt(tiles))
    if side * side != tiles:
        raise ValueError(f"expected square tile count, got {tiles}")
    rows = [torch.cat([images[row * side + column] for column in range(side)], dim=1) for row in range(side)]
    return torch.cat(rows, dim=0).numpy()


def build_tiny_image_frame(
    outer_iteration: int,
    actual_endpoints: torch.Tensor,
    target_endpoints: torch.Tensor,
    conditions: torch.Tensor,
    metrics: Dict[str, object],
    display_samples_per_condition: int,
) -> np.ndarray:
    """Build a separately subsampled fixed-scale grid for one outer iteration."""
    if (
        isinstance(display_samples_per_condition, bool)
        or not isinstance(display_samples_per_condition, int)
        or display_samples_per_condition <= 0
    ):
        raise ValueError(
            f"expected positive integer display_samples_per_condition, got "
            f"{display_samples_per_condition!r}"
        )
    display_side = math.isqrt(display_samples_per_condition)
    if display_side * display_side != display_samples_per_condition:
        raise ValueError(
            f"display_samples_per_condition must be a perfect square, got "
            f"{display_samples_per_condition}"
        )
    condition_indices = conditions.argmax(dim=1)
    fig, axes = plt.subplots(3, 2, figsize=(8, 6), dpi=80, constrained_layout=True)
    for condition_index in range(3):
        mask = condition_indices == condition_index
        target_montage = _tiny_image_montage(
            target_endpoints[mask], tiles=display_samples_per_condition
        )
        actual_montage = _tiny_image_montage(
            actual_endpoints[mask], tiles=display_samples_per_condition
        )
        axes[condition_index, 0].imshow(target_montage, cmap="gray", vmin=0.0, vmax=1.0)
        axes[condition_index, 1].imshow(actual_montage, cmap="gray", vmin=0.0, vmax=1.0)
        axes[condition_index, 0].set_title(f"condition {condition_index}: empirical current target\nold + teacher branches")
        axes[condition_index, 1].set_title(
            f"condition {condition_index}: actual student\n"
            f"occ L1={metrics['target_relative_occupancy_l1'][condition_index]:.2f}, "
            f"missing={metrics['missing_resolvable_modes'][condition_index]}, "
            f"bg={metrics['actual_background_mass'][condition_index]:.2f}"
        )
        for axis in axes[condition_index]:
            axis.axis("off")
    fig.suptitle(
        f"Tiny conditional image outer iteration {outer_iteration + 1} — fixed pixel scale [0, 1]"
    )
    frame = figure_to_rgb(fig)
    plt.close(fig)
    if frame.dtype != np.uint8 or frame.ndim != 3 or frame.shape[2] != 3:
        raise RuntimeError(f"expected uint8 RGB frame, got dtype={frame.dtype}, shape={frame.shape}")
    return frame


def run_tiny_image_outer_loop(
    teacher: ConditionalImageGaussianTeacher,
    alpha: float,
    outer_iterations: int,
    cache_size: int,
    inner_updates: int,
    batch_size: int,
    ode_steps: int,
    student_width: int,
    learning_rate: float,
    seed: int,
    metric_samples_per_condition: int,
    display_samples_per_condition: int,
    export_gif: bool,
) -> TinyImageOuterResult:
    """Run condition-aware branch-once fitting and independently sized evaluation/display."""
    for name, value in (
        ("outer_iterations", outer_iterations),
        ("cache_size", cache_size),
        ("inner_updates", inner_updates),
        ("batch_size", batch_size),
        ("ode_steps", ode_steps),
        ("student_width", student_width),
        ("metric_samples_per_condition", metric_samples_per_condition),
        ("display_samples_per_condition", display_samples_per_condition),
    ):
        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
            raise ValueError(f"expected positive integer {name}, got {value!r}")
    if metric_samples_per_condition < 16:
        raise ValueError(
            f"expected metric_samples_per_condition>=16, got {metric_samples_per_condition}"
        )
    if display_samples_per_condition > metric_samples_per_condition:
        raise ValueError(
            f"display_samples_per_condition({display_samples_per_condition}) cannot exceed "
            f"metric_samples_per_condition({metric_samples_per_condition})"
        )
    if math.isqrt(display_samples_per_condition) ** 2 != display_samples_per_condition:
        raise ValueError(
            f"display_samples_per_condition must be a perfect square, got "
            f"{display_samples_per_condition}"
        )
    if not isinstance(export_gif, bool):
        raise TypeError(f"expected bool export_gif, got {type(export_gif).__name__}")
    started_at = time.perf_counter()
    initialization_seed = cpu_generator(seed, 905).initial_seed()
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(initialization_seed)
        current_student = VelocityMLP(state_dim=64, condition_dim=3, width=student_width)
    current_student = clone_frozen_model(current_student)
    records: List[TinyImageOuterRecord] = []
    metric_count = 3 * metric_samples_per_condition
    for outer_index in range(outer_iterations):
        old_student = clone_frozen_model(current_student)
        cache = generate_conditional_image_cache(
            old_student,
            teacher,
            cache_size,
            ode_steps,
            alpha,
            seed + 10_000 * outer_index + 1,
        )
        fitted_student, losses, _ = fit_conditional_image_student(
            old_student,
            cache,
            inner_updates,
            batch_size,
            learning_rate,
            seed + 10_000 * outer_index + 2,
        )
        current_student = clone_frozen_model(fitted_student)
        conditions = balanced_tiny_conditions(metric_count, seed + 10_000 * outer_index + 3)
        metric_base = teacher.sample_base(metric_count, cpu_generator(seed, 10_000 * outer_index + 4))
        with torch.no_grad():
            old_endpoints = integrate_fixed_rk4(old_student, metric_base, ode_steps, conditions)[1][:, -1]
            actual_endpoints = integrate_fixed_rk4(current_student, metric_base, ode_steps, conditions)[1][:, -1]
        teacher_endpoints, _ = teacher.sample_endpoint(
            conditions, cpu_generator(seed, 10_000 * outer_index + 5)
        )
        target_teacher_branch = torch.rand(
            metric_count, generator=cpu_generator(seed, 10_000 * outer_index + 6)
        ) < float(alpha)
        target_endpoints = old_endpoints.clone()
        target_endpoints[target_teacher_branch] = teacher_endpoints[target_teacher_branch]
        metrics = tiny_image_metrics(actual_endpoints, target_endpoints, conditions)
        frame = build_tiny_image_frame(
            outer_index,
            actual_endpoints,
            target_endpoints,
            conditions,
            metrics,
            display_samples_per_condition,
        )
        records.append(
            TinyImageOuterRecord(
                outer_iteration=outer_index,
                old_student=old_student,
                student=current_student,
                cache=cache,
                actual_endpoints=actual_endpoints.detach(),
                target_endpoints=target_endpoints.detach(),
                conditions=conditions.detach(),
                target_teacher_branch=target_teacher_branch.detach(),
                losses=losses,
                metrics=metrics,
                frame=frame,
            )
        )
    gif_path: Optional[Path] = None
    if export_gif:
        frames = [record.frame for record in records]
        if not frames or any(frame.shape != frames[0].shape for frame in frames):
            raise RuntimeError("tiny image GIF requires non-empty frames with identical shapes")
        saved_paths = validated_save_frames(frames, "tiny_image_outer", CONFIG.export_fps)
        gif_path = saved_paths["gif"]
        gif_info = inspect_gif_timing(gif_path)
        if gif_info["frame_count"] != len(frames):
            raise RuntimeError(
                f"tiny image GIF frame mismatch: expected {len(frames)}, got {gif_info['frame_count']}"
            )
        if len(frames) > 1 and (
            gif_info["mean_frame_duration_seconds"] is None
            or abs(gif_info["mean_frame_duration_seconds"] - 1.0 / CONFIG.export_fps) > 0.02
        ):
            raise RuntimeError(
                f"tiny image GIF timing mismatch for fps={CONFIG.export_fps}: {gif_info!r}"
            )
    return TinyImageOuterResult(
        final_student=clone_frozen_model(current_student),
        records=records,
        runtime_seconds=time.perf_counter() - started_at,
        gif_path=gif_path,
    )

### What this analogy does and does not establish

A deterministic ODE maps each initial point to one endpoint. Under the usual regularity assumptions its flow preserves full support/topological connectivity, so visually distinct modes here are separated **high-density regions**, not disconnected mathematical support. The Gaussian teacher likewise has positive density everywhere. A finite-step RK4 solution and a finite-width MLP add approximation error; the generated distribution is measured rather than declared equal to the current mixture.

Pixel-space Euclidean distance on 8×8 arrays is only a convenient diagnostic. It treats a one-pixel translation as very different and does not model semantic equivalence, natural-image statistics, or decoder artifacts. To avoid forcing arbitrary noise into a semantic mode, nearest-template assignment uses per-pixel RMSE and accepts a point only within a mode-specific threshold of `3 × teacher endpoint sigma` by default; all other points enter an explicit background/unassigned bin. Occupancy, coverage, and rare recall exclude that background while reporting its mass separately.

Coverage is target-relative: a mode is evaluated only when the empirical current target contains at least five assigned examples, and the student must recover at least half of that target count. A rare mode below this count is reported as **unresolved**, with recall `None`, never as an automatic success. This matters because with `alpha=0.25`, a 3% teacher mode has only `0.75%` unconditional expected mass before any old-student contribution; a fixed 1% rule cannot resolve it reliably.

Metric sampling is therefore configured independently from visualization. `tiny_metric_samples_per_condition` defaults to 1536 (2048 in full mode), giving an expected `1536 × 0.25 × 0.03 = 11.52` teacher-rare examples per condition in the current mixture—above the five-example resolution floor. `tiny_display_samples_per_condition` remains 16, so sample grids stay compact regardless of metric sample count. The default smoke intentionally uses only 24 metric samples per condition and asserts that rare recall is unresolved; the larger resolution check and full run remain opt-in.

Pairwise pixel diversity is noise-sensitive: Gaussian noise can increase it without adding semantic variety. The notebook therefore reports actual-minus-target diversity together with background mass. FFT high-frequency energy excludes the DC coefficient from its denominator and is also reported as an actual-minus-target gap; neither metric replaces perceptual evaluation. Real image-generation complexity depends strongly on the VAE representation, prompt distribution and conditioning encoder, denoiser architecture, and perceptual metrics.

Most importantly, every outer iteration evaluates the student against the empirical current target `(1-alpha) * old student + alpha * teacher`. Its occupancy target therefore includes the frozen old-student branch. Raw teacher weights are shown only when defining the teacher and are never used as the claimed target fit after the outer loop begins.

In [ ]:
_tiny_smoke_started_at = time.perf_counter()
TINY_IMAGE_SMOKE_RESULT = run_tiny_image_outer_loop(
    teacher=TINY_IMAGE_TEACHER,
    alpha=CONFIG.alpha,
    outer_iterations=1,
    cache_size=96,
    inner_updates=2,
    batch_size=24,
    ode_steps=4,
    student_width=CONFIG.tiny_student_width,
    learning_rate=CONFIG.learning_rate,
    seed=CONFIG.seed + 30_000,
    metric_samples_per_condition=24,
    display_samples_per_condition=16,
    export_gif=False,
)
_smoke_record = TINY_IMAGE_SMOKE_RESULT.records[0]
_smoke_condition_indices = _smoke_record.cache.conditions.argmax(dim=1)
_smoke_branch_condition_counts = {
    f"condition_{condition_index}_{branch_name}": int(
        ((_smoke_condition_indices == condition_index) & branch_mask).sum()
    )
    for condition_index in range(3)
    for branch_name, branch_mask in (
        ("old", ~_smoke_record.cache.teacher_branch),
        ("teacher", _smoke_record.cache.teacher_branch),
    )
}
if any(count <= 0 for count in _smoke_branch_condition_counts.values()):
    raise RuntimeError(
        f"tiny smoke expected old and teacher branches for every condition, got {_smoke_branch_condition_counts!r}"
    )

_smoke_target_condition_indices = _smoke_record.conditions.argmax(dim=1)
_smoke_target_branch_condition_counts = {
    f"condition_{condition_index}_{branch_name}": int(
        ((_smoke_target_condition_indices == condition_index) & branch_mask).sum()
    )
    for condition_index in range(3)
    for branch_name, branch_mask in (
        ("old", ~_smoke_record.target_teacher_branch),
        ("teacher", _smoke_record.target_teacher_branch),
    )
}
if any(count <= 0 for count in _smoke_target_branch_condition_counts.values()):
    raise RuntimeError(
        f"tiny smoke empirical target expected both branches per condition, got "
        f"{_smoke_target_branch_condition_counts!r}"
    )

_smoke_label_max_errors: Dict[str, float] = {}
for branch_name, branch_mask, velocity_model in (
    ("old", ~_smoke_record.cache.teacher_branch, _smoke_record.old_student),
    ("teacher", _smoke_record.cache.teacher_branch, TINY_IMAGE_TEACHER.velocity),
):
    selected_indices = torch.where(branch_mask)[0][:4]
    selected_states = _smoke_record.cache.states[selected_indices].reshape(-1, 64)
    selected_conditions = _smoke_record.cache.conditions[selected_indices].repeat_interleave(
        _smoke_record.cache.times.numel(), dim=0
    )
    selected_times = _smoke_record.cache.times.repeat(selected_indices.numel())
    selected_labels = _smoke_record.cache.velocities[selected_indices].reshape(-1, 64)
    with torch.no_grad():
        recomputed_labels = velocity_model(selected_times, selected_states, selected_conditions)
    label_max_error = float((recomputed_labels - selected_labels).abs().max())
    if not math.isfinite(label_max_error) or label_max_error > 1.0e-6:
        raise RuntimeError(
            f"tiny smoke {branch_name} cache labels disagree with recomputed source velocity: "
            f"max_error={label_max_error}"
        )
    _smoke_label_max_errors[branch_name] = label_max_error

_teacher_check_sample_count = 96 if MIXTURE_DENSITY_TEST_MODE else 192
_teacher_check_coarse_steps = 8
_teacher_check_conditions = balanced_tiny_conditions(
    _teacher_check_sample_count, CONFIG.seed + 30_100
)
_teacher_check_base = TINY_IMAGE_TEACHER.sample_base(
    _teacher_check_sample_count, cpu_generator(CONFIG.seed, 30_101)
)
with torch.no_grad():
    _teacher_coarse_endpoint = integrate_fixed_rk4(
        TINY_IMAGE_TEACHER.velocity,
        _teacher_check_base,
        _teacher_check_coarse_steps,
        _teacher_check_conditions,
    )[1][:, -1]
    _teacher_fine_endpoint = integrate_fixed_rk4(
        TINY_IMAGE_TEACHER.velocity,
        _teacher_check_base,
        2 * _teacher_check_coarse_steps,
        _teacher_check_conditions,
    )[1][:, -1]
_teacher_step_doubling_rmse = float(
    torch.sqrt((_teacher_coarse_endpoint - _teacher_fine_endpoint).square().mean())
)
_teacher_step_doubling_tolerance = 0.05
if not math.isfinite(_teacher_step_doubling_rmse) or _teacher_step_doubling_rmse > _teacher_step_doubling_tolerance:
    raise RuntimeError(
        f"conditional teacher RK4 step-doubling RMSE exceeded tolerance: "
        f"error={_teacher_step_doubling_rmse}, tolerance={_teacher_step_doubling_tolerance}"
    )
_teacher_direct_endpoint, _ = TINY_IMAGE_TEACHER.sample_endpoint(
    _teacher_check_conditions, cpu_generator(CONFIG.seed, 30_102)
)
_teacher_condition_indices = _teacher_check_conditions.argmax(dim=1)
_teacher_mean_rmse = []
_teacher_second_moment_gaps = []
for condition_index in range(3):
    condition_mask = _teacher_condition_indices == condition_index
    integrated_samples = _teacher_fine_endpoint[condition_mask]
    direct_samples = _teacher_direct_endpoint[condition_mask]
    _teacher_mean_rmse.append(
        float(torch.sqrt((integrated_samples.mean(dim=0) - direct_samples.mean(dim=0)).square().mean()))
    )
    _teacher_second_moment_gaps.append(
        float((integrated_samples.square().mean() - direct_samples.square().mean()).abs())
    )
_teacher_mean_rmse_tolerance = 0.35
_teacher_second_moment_tolerance = 0.20
if max(_teacher_mean_rmse) > _teacher_mean_rmse_tolerance or max(_teacher_second_moment_gaps) > _teacher_second_moment_tolerance:
    raise RuntimeError(
        f"conditional teacher RK4 endpoint disagrees with analytic endpoint samples: "
        f"mean_rmse={_teacher_mean_rmse}, mean_tolerance={_teacher_mean_rmse_tolerance}, "
        f"second_moment_gaps={_teacher_second_moment_gaps}, "
        f"second_moment_tolerance={_teacher_second_moment_tolerance}"
    )

_expected_cache_shape = (96, 5, 64)
_expected_endpoint_shape = (72, 64)
if _smoke_record.cache.states.shape != _expected_cache_shape:
    raise RuntimeError(
        f"tiny smoke expected cache shape {_expected_cache_shape}, got {tuple(_smoke_record.cache.states.shape)}"
    )
if _smoke_record.actual_endpoints.shape != _expected_endpoint_shape or _smoke_record.target_endpoints.shape != _expected_endpoint_shape:
    raise RuntimeError(
        f"tiny smoke expected actual/target shape {_expected_endpoint_shape}, got "
        f"{tuple(_smoke_record.actual_endpoints.shape)}, {tuple(_smoke_record.target_endpoints.shape)}"
    )
if len(_smoke_record.losses) != 2 or not all(math.isfinite(loss) for loss in _smoke_record.losses):
    raise RuntimeError(f"tiny smoke expected two finite losses, got {_smoke_record.losses!r}")
if _smoke_record.metrics["rare_mode_resolvable"] != [False, False, False]:
    raise RuntimeError(
        f"tiny smoke rare mode must be unresolved at 24 samples per condition, got "
        f"{_smoke_record.metrics['rare_mode_resolvable']!r}"
    )
if _smoke_record.metrics["rare_mode_recall"] != [None, None, None]:
    raise RuntimeError(
        f"unresolved tiny-smoke rare recall must be None rather than success, got "
        f"{_smoke_record.metrics['rare_mode_recall']!r}"
    )
_smoke_metric_values: List[float] = []
for metric_name, metric_value in _smoke_record.metrics.items():
    pending_values = metric_value if isinstance(metric_value, list) else [metric_value]
    flattened_values: List[object] = []
    for pending_value in pending_values:
        if isinstance(pending_value, list):
            flattened_values.extend(pending_value)
        else:
            flattened_values.append(pending_value)
    for scalar_value in flattened_values:
        if scalar_value is None:
            continue
        if not isinstance(scalar_value, (bool, int, float)) or not math.isfinite(float(scalar_value)):
            raise RuntimeError(
                f"tiny smoke metric {metric_name!r} contains invalid numeric value: {scalar_value!r}"
            )
        _smoke_metric_values.append(float(scalar_value))
if _smoke_record.frame.dtype != np.uint8 or _smoke_record.frame.shape != (480, 640, 3):
    raise RuntimeError(
        f"tiny smoke expected uint8 frame shape (480, 640, 3), got "
        f"dtype={_smoke_record.frame.dtype}, shape={_smoke_record.frame.shape}"
    )
TINY_IMAGE_SMOKE_SUMMARY = {
    "runtime_seconds": time.perf_counter() - _tiny_smoke_started_at,
    "cache_shape": tuple(_smoke_record.cache.states.shape),
    "endpoint_shape": tuple(_smoke_record.actual_endpoints.shape),
    "condition_counts": _smoke_record.cache.conditions.sum(dim=0).int().tolist(),
    "source_branch_condition_counts": _smoke_branch_condition_counts,
    "target_branch_condition_counts": _smoke_target_branch_condition_counts,
    "source_label_max_errors": _smoke_label_max_errors,
    "teacher_step_doubling_rmse": _teacher_step_doubling_rmse,
    "teacher_step_doubling_tolerance": _teacher_step_doubling_tolerance,
    "teacher_endpoint_mean_rmse": _teacher_mean_rmse,
    "teacher_endpoint_mean_rmse_tolerance": _teacher_mean_rmse_tolerance,
    "teacher_endpoint_second_moment_gaps": _teacher_second_moment_gaps,
    "teacher_endpoint_second_moment_tolerance": _teacher_second_moment_tolerance,
    "assignment_sigma_multiplier": _smoke_record.metrics["assignment_sigma_multiplier"],
    "assignment_rmse_thresholds": _smoke_record.metrics["assignment_rmse_thresholds"],
    "minimum_resolvable_count": _smoke_record.metrics["minimum_resolvable_count"],
    "actual_background_mass": _smoke_record.metrics["actual_background_mass"],
    "target_background_mass": _smoke_record.metrics["target_background_mass"],
    "rare_mode_resolvable": _smoke_record.metrics["rare_mode_resolvable"],
    "rare_mode_recall": _smoke_record.metrics["rare_mode_recall"],
    "teacher_branches": int(_smoke_record.cache.teacher_branch.sum()),
    "old_student_branches": int((~_smoke_record.cache.teacher_branch).sum()),
    "losses": _smoke_record.losses,
    "finite_gradients": True,
    "finite_metric_value_count": len(_smoke_metric_values),
    "frame_shape": _smoke_record.frame.shape,
    "finite": True,
}
print(TINY_IMAGE_SMOKE_SUMMARY)

# The larger sampling check is automatic in test mode and remains opt-in for readers.
# It isolates rare-mode count semantics from the much slower outer-loop and pairwise metrics.
RUN_TINY_METRIC_RESOLUTION_CHECK = MIXTURE_DENSITY_TEST_MODE
TINY_IMAGE_METRIC_RESOLUTION_SUMMARY = None
if RUN_TINY_METRIC_RESOLUTION_CHECK:
    _metric_resolution_started_at = time.perf_counter()
    _resolution_sample_count = 3 * CONFIG.tiny_metric_samples_per_condition
    _resolution_conditions = balanced_tiny_conditions(
        _resolution_sample_count, CONFIG.seed + 35_000
    )
    _resolution_old_endpoints = TINY_IMAGE_TEACHER.sample_base(
        _resolution_sample_count, cpu_generator(CONFIG.seed, 35_001)
    )
    _resolution_teacher_endpoints, _ = TINY_IMAGE_TEACHER.sample_endpoint(
        _resolution_conditions, cpu_generator(CONFIG.seed, 35_002)
    )
    _resolution_teacher_branch = torch.rand(
        _resolution_sample_count, generator=cpu_generator(CONFIG.seed, 35_003)
    ) < CONFIG.alpha
    _resolution_target_endpoints = _resolution_old_endpoints.clone()
    _resolution_target_endpoints[_resolution_teacher_branch] = _resolution_teacher_endpoints[
        _resolution_teacher_branch
    ]
    _resolution_assignment = tiny_image_mode_occupancy(
        _resolution_target_endpoints,
        _resolution_conditions,
        TINY_TEMPLATE_MEANS,
        TINY_IMAGE_TEACHER.widths,
    )
    _resolution_rare_counts = _resolution_assignment["counts"][:, 3].tolist()
    _resolution_minimum_count = int(_smoke_record.metrics["minimum_resolvable_count"])
    _resolution_flags = [count >= _resolution_minimum_count for count in _resolution_rare_counts]
    if _resolution_flags != [True, True, True]:
        raise RuntimeError(
            "configured metric sample count must resolve the empirical rare mode for every "
            f"condition, got flags={_resolution_flags!r}, target_rare_counts="
            f"{_resolution_rare_counts!r}, minimum_count={_resolution_minimum_count}"
        )
    TINY_IMAGE_METRIC_RESOLUTION_SUMMARY = {
        "runtime_seconds": time.perf_counter() - _metric_resolution_started_at,
        "metric_samples_per_condition": CONFIG.tiny_metric_samples_per_condition,
        "display_samples_per_condition": CONFIG.tiny_display_samples_per_condition,
        "minimum_resolvable_count": _resolution_minimum_count,
        "target_rare_counts": _resolution_rare_counts,
        "rare_mode_resolvable": _resolution_flags,
        "target_background_mass": _resolution_assignment["background_mass"].tolist(),
    }
    print(TINY_IMAGE_METRIC_RESOLUTION_SUMMARY)

TINY_IMAGE_FULL_RESULT = None
if CONFIG.run_tiny_image:
    TINY_IMAGE_FULL_RESULT = run_tiny_image_outer_loop(
        teacher=TINY_IMAGE_TEACHER,
        alpha=CONFIG.alpha,
        outer_iterations=CONFIG.tiny_outer_iterations,
        cache_size=CONFIG.tiny_cache_size,
        inner_updates=CONFIG.tiny_inner_updates,
        batch_size=CONFIG.tiny_batch_size,
        ode_steps=CONFIG.tiny_ode_steps,
        student_width=CONFIG.tiny_student_width,
        learning_rate=CONFIG.learning_rate,
        seed=CONFIG.seed + 40_000,
        metric_samples_per_condition=CONFIG.tiny_metric_samples_per_condition,
        display_samples_per_condition=CONFIG.tiny_display_samples_per_condition,
        export_gif=True,
    )
    _full_frames = [record.frame for record in TINY_IMAGE_FULL_RESULT.records]
    if len(_full_frames) != CONFIG.tiny_outer_iterations:
        raise RuntimeError(
            f"expected {CONFIG.tiny_outer_iterations} full tiny-image frames, got {len(_full_frames)}"
        )
    if TINY_IMAGE_FULL_RESULT.gif_path is None or not TINY_IMAGE_FULL_RESULT.gif_path.is_file():
        raise RuntimeError(f"full tiny-image run did not produce a GIF: {TINY_IMAGE_FULL_RESULT.gif_path!r}")
    for record in TINY_IMAGE_FULL_RESULT.records:
        if record.frame.dtype != np.uint8 or record.frame.shape != _full_frames[0].shape:
            raise RuntimeError(
                f"full tiny-image frame validation failed at outer {record.outer_iteration}: "
                f"dtype={record.frame.dtype}, shape={record.frame.shape}"
            )
        for metric_name, metric_value in record.metrics.items():
            pending_values = metric_value if isinstance(metric_value, list) else [metric_value]
            for pending_value in pending_values:
                scalar_values = pending_value if isinstance(pending_value, list) else [pending_value]
                for scalar_value in scalar_values:
                    if scalar_value is None:
                        continue
                    if not isinstance(scalar_value, (bool, int, float)) or not math.isfinite(float(scalar_value)):
                        raise RuntimeError(
                            f"full tiny-image metric {metric_name!r} is invalid at outer "
                            f"{record.outer_iteration}: {scalar_value!r}"
                        )
    print(
        {
            "runtime_seconds": TINY_IMAGE_FULL_RESULT.runtime_seconds,
            "outer_iterations": len(TINY_IMAGE_FULL_RESULT.records),
            "frame_shape": _full_frames[0].shape,
            "gif_path": TINY_IMAGE_FULL_RESULT.gif_path,
            "gif_timing": inspect_gif_timing(TINY_IMAGE_FULL_RESULT.gif_path),
            "final_metrics": TINY_IMAGE_FULL_RESULT.records[-1].metrics,
        }
    )

TOTAL_NOTEBOOK_RUNTIME_SECONDS = time.perf_counter() - NOTEBOOK_STARTED_AT
NOTEBOOK_RUNTIME_SUMMARY = {
    "total_notebook_runtime_seconds": TOTAL_NOTEBOOK_RUNTIME_SECONDS,
    "main_2d_runtime_seconds": (
        None if TRAINING_2D_ARTIFACTS is None else TRAINING_2D_ARTIFACTS["runtime_seconds"]
    ),
    "test_mode": MIXTURE_DENSITY_TEST_MODE,
}
RUNTIME_SUMMARY_PATH = OUTPUT_DIR / "runtime_summary.json"
with RUNTIME_SUMMARY_PATH.open("w", encoding="utf-8") as handle:
    json.dump(_json_ready(NOTEBOOK_RUNTIME_SUMMARY), handle, indent=2)
if not RUNTIME_SUMMARY_PATH.is_file() or RUNTIME_SUMMARY_PATH.stat().st_size <= 0:
    raise RuntimeError(f"runtime summary JSON was not written: {RUNTIME_SUMMARY_PATH}")
if TRAINING_2D_ARTIFACTS is not None:
    TRAINING_2D_ARTIFACTS["total_notebook_runtime_seconds"] = TOTAL_NOTEBOOK_RUNTIME_SECONDS
    TRAINING_2D_ARTIFACTS["runtime_summary_path"] = RUNTIME_SUMMARY_PATH
print(_json_ready(NOTEBOOK_RUNTIME_SUMMARY))

## Flow-Factory mapping and practical run guide

The notebook's **one branch per complete trajectory** is the presentation-level analogue of Flow-Factory's `xopd_target_mode: marginal_cfm`. Set `marginal_cfm_alpha` to the teacher-branch probability, keep `xopd_dk_space: v` and `normalize_d_k: false`, and use an `ODE` scheduler with `noise_level: 0.0`. In production, `latent_index_map` selects stored trajectory states, `callback_index_map` selects the corresponding detached `noise_pred` source velocities, and `marginal_cfm_branch` records whether each row followed the old student or teacher.

The main optimization curves remain `train/loss` and `train/d_k`. Marginal-CFM diagnostics live under `train/marginal_cfm/`: `callback_count`, `teacher_branch_fraction_{min,max,mean,std}`, branch-conditioned `loss_old_*` and `loss_teacher_*`, `target_velocity_rms_*`, `target_velocity_l2_*`, and `student_target_gap_rms_*`. There is intentionally no transition-responsibility `gamma` log because this method never evaluates a density ratio.

Run this notebook from the repository root. The normal order is: imports/configuration → analytic 2D teacher and core smoke → full FAST_MODE 2D outer training → visual smoke → conditional 8×8 teacher and tiny-image smoke → optional tiny-image full run. Set `MIXTURE_DENSITY_TEST_MODE=1` only for automated checks; it skips the expensive full 2D animation while retaining the core, visual, tiny-image, and rare-mode-resolution checks. Set `CONFIG.run_tiny_image` through an explicitly constructed config or call `run_tiny_image_outer_loop` to produce the larger conditional-image experiment.

Run the complete notebook validation, including normal FAST_MODE artifact generation, from a clean isolated environment with exactly:

```bash
uv run --isolated --frozen --with pytest --with matplotlib pytest tests/test_mixture_of_density_demo.py
```

The isolated command supplies the test runner and plotting dependency without updating the project environment or lockfile.

All generated files are under `.scratch/mixture_of_density_demo/`. The normal 2D run writes `outer_training.gif`/`.mp4`, `generation_trajectories.gif`/`.mp4`, `wrong_controls_comparison.png`, `endpoint_matched_kde_responsibility.png`, `metrics_2d.json`, and `config.json`; smoke mode also writes `visual_smoke.gif` and may write `visual_smoke.mp4` when FFmpeg is available. The optional tiny-image export writes `tiny_image_outer.gif` and may write its MP4 companion.

Limitations are deliberate: this is a CPU pedagogical model, not a DiT/VAE reproduction; fixed-grid KDE, finite projections, tiny-image pixel metrics, finite RK4 steps, finite samples, and a narrow MLP all introduce approximation error. The deterministic student matches the target time marginals only in the ideal population/capacity limit, not the random source path law. Production validation still requires the documented multi-GPU XOPD smoke because this notebook does not exercise distributed collectives, real model conditioning, mixed precision, or accelerator wrappers.